# 08 — SPY Option-Chain Ingestion and Quote Filtering

## Purpose

This notebook builds the empirical data layer for the v1.1 derivatives-pricing sequence.

The goal is to ingest SPY option-chain data, standardize calls and puts into one auditable quote panel, apply quote-quality filters, match call-put pairs by expiry and strike, and export a clean candidate dataset for downstream implied-volatility and static-arbitrage diagnostics.

This notebook does **not** compute implied volatility, fit volatility surfaces, repair arbitrage, calibrate Heston/Bates models, compute Greeks, or run hedging diagnostics.

## Position in the v1.1 sequence

Previous notebooks established the controlled theoretical foundation:

1. binomial no-arbitrage replication
2. Brownian motion, GBM, and quadratic variation
3. Ito's lemma and the Black-Scholes PDE
4. Black-Scholes formula, Greeks, and numerical checks
5. risk-neutral Monte Carlo and delta-hedging error
6. implied-volatility inversion and pointwise no-arbitrage bounds
7. static-arbitrage diagnostics for synthetic option surfaces

Notebook 08 is the transition from synthetic theory to empirical option-chain data.

Its job is not to prove that the market surface is clean. Its job is to prepare a quote-quality-controlled dataset so that Notebook 09 can test the raw market surface properly.

## Main question

Can raw SPY option-chain data be ingested, standardized, filtered, paired, and documented well enough to support downstream implied-volatility and static-arbitrage diagnostics?

## Key principle

Data validity comes before model validity.

A bad option-chain panel can make any pricing model, implied-volatility surface, or calibration routine look misleading. This notebook therefore focuses on reproducible data handling and quote-quality discipline before any pricing claims are made.

## Expected outputs

This notebook should produce:

1. a raw standardized option quote panel
2. a quote-quality-filtered option quote panel
3. a rejection ledger explaining removed rows
4. a matched call-put panel by expiry and strike
5. expiry-level quote and parity diagnostics
6. a Notebook 09 candidate dataset
7. a reproducibility manifest
8. a final readiness flag

The final readiness flag is:

`NOTEBOOK_08_READY_FOR_09`

## Important limitations

SPY options are listed ETF options and may contain market microstructure issues, stale quotes, wide spreads, dividend effects, and American-exercise complications.

This notebook treats SPY option chains as an empirical stress test for the diagnostic framework, not as a perfectly clean European option laboratory.

Vendor-provided implied volatility may be stored for comparison, but it is not treated as authoritative. Downstream notebooks will compute implied volatility independently from filtered option prices.

## What this notebook must not do

This notebook must not:

- compute final implied volatility
- construct a volatility surface
- fit SVI or SSVI
- repair static arbitrage
- calibrate Heston or Bates models
- compute Greeks
- run stress tests
- claim that the market surface is arbitrage-free

Those tasks belong to later notebooks.

In [1]:
# ============================================================
# 08 — SPY Option-Chain Ingestion and Quote Filtering
# Cell 2: Imports, configuration, paths, and reproducibility
# ============================================================

from __future__ import annotations

import json
import math
import platform
import sys
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    import yfinance as yf
    YFINANCE_AVAILABLE = True
except ImportError:
    yf = None
    YFINANCE_AVAILABLE = False


# ------------------------------------------------------------
# Notebook identity
# ------------------------------------------------------------

NOTEBOOK_ID = "08_spy_option_chain_ingestion_and_quote_filtering"
NOTEBOOK_VERSION = "v1.1"
UNDERLYING_TICKER = "SPY"

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc)
RUN_ID = RUN_TIMESTAMP_UTC.strftime("%Y%m%d_%H%M%S_UTC")


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
PROCESSED_DATA_DIR = DATA_ROOT / "processed"
MANIFEST_DIR = DATA_ROOT / "manifest"

for path in [DATA_ROOT, RAW_DATA_DIR, PROCESSED_DATA_DIR, MANIFEST_DIR]:
    path.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Quote-filtering configuration
# ------------------------------------------------------------

@dataclass(frozen=True)
class Notebook08Config:
    ticker: str = "SPY"
    data_source: str = "yfinance"
    snapshot_policy: str = "pull_then_cache"
    
    # Expiry selection
    min_dte_calendar: int = 1
    max_dte_calendar: int = 90
    max_expiries: int = 12
    
    # Basic quote-quality thresholds
    min_mid_price: float = 0.01
    max_abs_spread: float = 5.00
    max_rel_spread_mid: float = 0.50
    
    # Optional liquidity filters.
    # These are recorded but should not be too aggressive in the first pass.
    min_open_interest: int = 0
    min_volume: int = 0
    
    # Downstream eligibility thresholds for Notebook 09
    min_clean_calls_per_expiry: int = 7
    min_clean_puts_per_expiry: int = 7
    min_matched_pairs_per_expiry: int = 7
    
    # Numerical tolerances
    quote_tol: float = 1e-12
    parity_soft_tol_abs: float = 0.25
    
    # Calendar convention for initial year fraction.
    # Later notebooks can replace this with a more precise trading-calendar convention.
    year_basis: float = 365.0


CONFIG = Notebook08Config(ticker=UNDERLYING_TICKER)


# ------------------------------------------------------------
# Output artifact paths
# ------------------------------------------------------------

ARTIFACT_PATHS = {
    "raw_snapshot": RAW_DATA_DIR / f"{CONFIG.ticker.lower()}_option_chain_raw_snapshot_{RUN_ID}.parquet",
    "raw_standardized_panel": PROCESSED_DATA_DIR / f"{CONFIG.ticker.lower()}_option_quote_panel_raw_{RUN_ID}.parquet",
    "quote_filtered_panel": PROCESSED_DATA_DIR / f"{CONFIG.ticker.lower()}_option_quote_panel_filtered_{RUN_ID}.parquet",
    "matched_pairs": PROCESSED_DATA_DIR / f"{CONFIG.ticker.lower()}_option_matched_pairs_{RUN_ID}.parquet",
    "n09_candidate_panel": PROCESSED_DATA_DIR / f"{CONFIG.ticker.lower()}_option_n09_candidate_{RUN_ID}.parquet",
    "rejection_ledger": PROCESSED_DATA_DIR / f"{CONFIG.ticker.lower()}_option_rejection_ledger_{RUN_ID}.csv",
    "expiry_summary": PROCESSED_DATA_DIR / f"{CONFIG.ticker.lower()}_option_expiry_summary_{RUN_ID}.csv",
    "snapshot_manifest": MANIFEST_DIR / f"{CONFIG.ticker.lower()}_option_snapshot_manifest_{RUN_ID}.json",
}


# ------------------------------------------------------------
# Reproducibility manifest starter
# ------------------------------------------------------------

MANIFEST: dict[str, Any] = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "run_id": RUN_ID,
    "run_timestamp_utc": RUN_TIMESTAMP_UTC.isoformat(),
    "ticker": CONFIG.ticker,
    "data_source": CONFIG.data_source,
    "snapshot_policy": CONFIG.snapshot_policy,
    "config": asdict(CONFIG),
    "artifact_paths": {key: str(value) for key, value in ARTIFACT_PATHS.items()},
    "environment": {
        "python_version": sys.version,
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "yfinance_available": YFINANCE_AVAILABLE,
    },
}


# ------------------------------------------------------------
# Display setup
# ------------------------------------------------------------

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)


print("Notebook 08 configuration initialized.")
print(f"Run ID: {RUN_ID}")
print(f"Ticker: {CONFIG.ticker}")
print(f"Data source: {CONFIG.data_source}")
print(f"yfinance available: {YFINANCE_AVAILABLE}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")

Notebook 08 configuration initialized.
Run ID: 20260705_154512_UTC
Ticker: SPY
Data source: yfinance
yfinance available: True
Raw data directory: d:\Derivative Pricing Project v1.0+\V1.1\data\raw
Processed data directory: d:\Derivative Pricing Project v1.0+\V1.1\data\processed


In [2]:
# ============================================================
# Cell 3: Acquire raw SPY option-chain snapshot and cache it
# ============================================================

def safe_write_dataframe(df: pd.DataFrame, preferred_path: Path) -> Path:
    """
    Save a DataFrame to parquet when available.
    If parquet support is unavailable, fall back to CSV and return the actual path.
    """
    preferred_path = Path(preferred_path)
    preferred_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        df.to_parquet(preferred_path, index=False)
        return preferred_path
    except Exception as parquet_error:
        fallback_path = preferred_path.with_suffix(".csv")
        df.to_csv(fallback_path, index=False)

        print(f"Parquet write failed for {preferred_path.name}.")
        print(f"Fallback CSV written instead: {fallback_path.name}")
        print(f"Parquet error type: {type(parquet_error).__name__}")

        return fallback_path


def get_underlying_spot(ticker_obj: Any, ticker: str) -> float:
    """
    Pull a reasonable current underlying price estimate.
    This is only snapshot metadata for Notebook 08.
    It is not treated as a final forward or pricing input.
    """
    spot_candidates: list[float] = []

    try:
        fast_info = ticker_obj.fast_info

        for key in ["last_price", "lastPrice", "regular_market_price", "regularMarketPrice"]:
            value = getattr(fast_info, key, None)

            if value is not None and np.isfinite(value) and value > 0:
                spot_candidates.append(float(value))
    except Exception:
        pass

    try:
        hist = ticker_obj.history(period="5d", auto_adjust=False)

        if not hist.empty and "Close" in hist.columns:
            close_value = float(hist["Close"].dropna().iloc[-1])

            if np.isfinite(close_value) and close_value > 0:
                spot_candidates.append(close_value)
    except Exception:
        pass

    if not spot_candidates:
        raise RuntimeError(f"Could not obtain a valid underlying spot estimate for {ticker}.")

    return float(spot_candidates[0])


def expiry_dte_calendar(expiry_str: str, valuation_date: datetime) -> int:
    """
    Calendar-day distance from valuation date to option expiry date.
    """
    expiry_date = pd.Timestamp(expiry_str).date()
    valuation_day = valuation_date.date()
    return int((expiry_date - valuation_day).days)


def select_expiries(
    available_expiries: list[str],
    valuation_date: datetime,
    min_dte: int,
    max_dte: int,
    max_expiries: int,
) -> pd.DataFrame:
    """
    Convert available expiry strings into an audited expiry-selection table.
    """
    expiry_rows = []

    for expiry in available_expiries:
        dte = expiry_dte_calendar(expiry, valuation_date)

        expiry_rows.append(
            {
                "expiry": expiry,
                "dte_calendar": dte,
                "passes_min_dte": dte >= min_dte,
                "passes_max_dte": dte <= max_dte,
            }
        )

    expiry_table = pd.DataFrame(expiry_rows)

    if expiry_table.empty:
        return expiry_table

    expiry_table["expiry_selected"] = (
        expiry_table["passes_min_dte"]
        & expiry_table["passes_max_dte"]
    )

    expiry_table = expiry_table.sort_values(["dte_calendar", "expiry"]).reset_index(drop=True)

    selected_index = expiry_table.index[expiry_table["expiry_selected"]].tolist()[:max_expiries]
    expiry_table["selected_for_pull"] = expiry_table.index.isin(selected_index)

    return expiry_table


def pull_option_chain_snapshot(config: Notebook08Config) -> tuple[pd.DataFrame, pd.DataFrame, float]:
    """
    Pull raw calls and puts for selected expiries from yfinance.

    Returns:
        raw_snapshot_df: one row per raw option quote
        expiry_selection_df: expiry audit table
        underlying_spot: current underlying price estimate
    """
    if not YFINANCE_AVAILABLE:
        raise ImportError("yfinance is not available. Install yfinance or load a cached snapshot.")

    ticker_obj = yf.Ticker(config.ticker)

    available_expiries = list(ticker_obj.options)

    if not available_expiries:
        raise RuntimeError(f"No option expiries returned by yfinance for {config.ticker}.")

    underlying_spot = get_underlying_spot(ticker_obj=ticker_obj, ticker=config.ticker)

    expiry_selection_df = select_expiries(
        available_expiries=available_expiries,
        valuation_date=RUN_TIMESTAMP_UTC,
        min_dte=config.min_dte_calendar,
        max_dte=config.max_dte_calendar,
        max_expiries=config.max_expiries,
    )

    selected_expiries = expiry_selection_df.loc[
        expiry_selection_df["selected_for_pull"], "expiry"
    ].tolist()

    if not selected_expiries:
        raise RuntimeError(
            "No expiries passed the current selection rules. "
            "Loosen min/max DTE or increase max_expiries."
        )

    raw_frames = []
    expiry_pull_records = []

    for expiry in selected_expiries:
        try:
            chain = ticker_obj.option_chain(expiry)

            calls = chain.calls.copy()
            puts = chain.puts.copy()

            calls["option_type"] = "call"
            puts["option_type"] = "put"

            calls["expiry"] = expiry
            puts["expiry"] = expiry

            calls["source"] = config.data_source
            puts["source"] = config.data_source

            calls["ticker"] = config.ticker
            puts["ticker"] = config.ticker

            calls["snapshot_ts_utc"] = RUN_TIMESTAMP_UTC.isoformat()
            puts["snapshot_ts_utc"] = RUN_TIMESTAMP_UTC.isoformat()

            calls["underlying_spot_snapshot"] = underlying_spot
            puts["underlying_spot_snapshot"] = underlying_spot

            raw_frames.extend([calls, puts])

            expiry_pull_records.append(
                {
                    "expiry": expiry,
                    "dte_calendar": expiry_dte_calendar(expiry, RUN_TIMESTAMP_UTC),
                    "call_rows": int(len(calls)),
                    "put_rows": int(len(puts)),
                    "total_rows": int(len(calls) + len(puts)),
                    "pull_status": "OK",
                    "error_type": None,
                    "error_message": None,
                }
            )

        except Exception as error:
            expiry_pull_records.append(
                {
                    "expiry": expiry,
                    "dte_calendar": expiry_dte_calendar(expiry, RUN_TIMESTAMP_UTC),
                    "call_rows": 0,
                    "put_rows": 0,
                    "total_rows": 0,
                    "pull_status": "FAILED",
                    "error_type": type(error).__name__,
                    "error_message": str(error),
                }
            )

    if not raw_frames:
        raise RuntimeError("No option-chain rows were successfully pulled.")

    raw_snapshot_df = pd.concat(raw_frames, ignore_index=True)

    expiry_pull_df = pd.DataFrame(expiry_pull_records)

    expiry_selection_df = expiry_selection_df.merge(
        expiry_pull_df,
        on=["expiry", "dte_calendar"],
        how="left",
    )

    return raw_snapshot_df, expiry_selection_df, underlying_spot


raw_option_snapshot, expiry_selection_summary, underlying_spot_snapshot = pull_option_chain_snapshot(CONFIG)

actual_raw_snapshot_path = safe_write_dataframe(
    raw_option_snapshot,
    ARTIFACT_PATHS["raw_snapshot"],
)

ARTIFACT_PATHS["raw_snapshot"] = actual_raw_snapshot_path
MANIFEST["artifact_paths"]["raw_snapshot"] = str(actual_raw_snapshot_path)

MANIFEST["underlying_spot_snapshot"] = float(underlying_spot_snapshot)
MANIFEST["available_expiry_count"] = int(len(expiry_selection_summary))
MANIFEST["selected_expiry_count"] = int(expiry_selection_summary["selected_for_pull"].sum())
MANIFEST["raw_snapshot_rows"] = int(len(raw_option_snapshot))
MANIFEST["raw_call_rows"] = int((raw_option_snapshot["option_type"] == "call").sum())
MANIFEST["raw_put_rows"] = int((raw_option_snapshot["option_type"] == "put").sum())
MANIFEST["selected_expiries"] = expiry_selection_summary.loc[
    expiry_selection_summary["selected_for_pull"], "expiry"
].tolist()

print("Raw option-chain snapshot acquired and cached.")
print(f"Underlying spot snapshot: {underlying_spot_snapshot:,.4f}")
print(f"Available expiries: {MANIFEST['available_expiry_count']}")
print(f"Selected expiries: {MANIFEST['selected_expiry_count']}")
print(f"Raw option rows: {MANIFEST['raw_snapshot_rows']:,}")
print(f"Raw call rows: {MANIFEST['raw_call_rows']:,}")
print(f"Raw put rows: {MANIFEST['raw_put_rows']:,}")
print(f"Raw snapshot path: {actual_raw_snapshot_path}")

display(expiry_selection_summary.head(20))
display(raw_option_snapshot.head())

Raw option-chain snapshot acquired and cached.
Underlying spot snapshot: 744.7800
Available expiries: 28
Selected expiries: 12
Raw option rows: 3,954
Raw call rows: 2,009
Raw put rows: 1,945
Raw snapshot path: d:\Derivative Pricing Project v1.0+\V1.1\data\raw\spy_option_chain_raw_snapshot_20260705_154512_UTC.parquet


,expiry,dte_calendar,passes_min_dte,passes_max_dte,expiry_selected,selected_for_pull,call_rows,put_rows,total_rows,pull_status,error_type,error_message
0,2026-07-06,1,True,True,True,True,99.000000,102.000000,201.000000,OK,None,None
1,2026-07-07,2,True,True,True,True,87.000000,87.000000,174.000000,OK,None,None
2,2026-07-08,3,True,True,True,True,83.000000,87.000000,170.000000,OK,None,None
3,2026-07-09,4,True,True,True,True,71.000000,75.000000,146.000000,OK,None,None
4,2026-07-10,5,True,True,True,True,192.000000,182.000000,374.000000,OK,None,None
5,2026-07-17,12,True,True,True,True,241.000000,209.000000,450.000000,OK,None,None
6,2026-07-24,19,True,True,True,True,169.000000,162.000000,331.000000,OK,None,None
7,2026-07-31,26,True,True,True,True,242.000000,227.000000,469.000000,OK,None,None
8,2026-08-07,33,True,True,True,True,94.000000,103.000000,197.000000,OK,None,None
9,2026-08-21,47,True,True,True,True,238.000000,209.000000,447.000000,OK,None,None


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,option_type,expiry,source,ticker,snapshot_ts_utc,underlying_spot_snapshot
0,SPY260706C00625000,2026-07-02 14:39:40+00:00,625.000000,123.710000,117.820000,121.340000,1.769997,1.451531,30.000000,31.000000,1.448001,True,REGULAR,USD,call,2026-07-06,yfinance,SPY,2026-07-05T15:45:12.973274+00:00,744.780029
1,SPY260706C00660000,2026-06-29 13:34:02+00:00,660.000000,87.600000,82.830000,86.350000,8.729996,11.068842,6.000000,26.000000,1.075688,True,REGULAR,USD,call,2026-07-06,yfinance,SPY,2026-07-05T15:45:12.973274+00:00,744.780029
2,SPY260706C00665000,2026-07-02 19:57:29+00:00,665.000000,80.060000,77.830000,81.350000,10.049995,14.355085,2.000000,0.000000,1.022222,True,REGULAR,USD,call,2026-07-06,yfinance,SPY,2026-07-05T15:45:12.973274+00:00,744.780029
3,SPY260706C00670000,2026-07-01 14:37:02+00:00,670.000000,75.090000,72.830000,76.350000,-1.910004,-2.480524,2.000000,12.000000,0.968994,True,REGULAR,USD,call,2026-07-06,yfinance,SPY,2026-07-05T15:45:12.973274+00:00,744.780029
4,SPY260706C00700000,2026-07-02 19:57:25+00:00,700.000000,45.140000,42.840000,46.360000,-1.729999,-3.691059,122.000000,5.000000,0.645755,True,REGULAR,USD,call,2026-07-06,yfinance,SPY,2026-07-05T15:45:12.973274+00:00,744.780029


In [3]:
# ============================================================
# Cell 4: Standardize raw option-chain panel and run integrity checks
# ============================================================

def standardize_raw_option_panel(
    raw_df: pd.DataFrame,
    config: Notebook08Config,
    valuation_ts: datetime,
) -> pd.DataFrame:
    """
    Convert the raw yfinance option-chain output into a stable long-format schema.

    This cell does not filter quotes yet.
    It only standardizes names, dtypes, timestamps, maturities, and traceability fields.
    """
    df = raw_df.copy()

    rename_map = {
        "contractSymbol": "contract_symbol",
        "lastTradeDate": "last_trade_date",
        "lastPrice": "last_price",
        "impliedVolatility": "vendor_implied_vol",
        "inTheMoney": "vendor_in_the_money",
        "contractSize": "contract_size",
    }

    df = df.rename(columns=rename_map)

    required_columns_with_defaults = {
        "snapshot_ts_utc": valuation_ts.isoformat(),
        "ticker": config.ticker,
        "source": config.data_source,
        "option_type": pd.NA,
        "expiry": pd.NA,
        "contract_symbol": pd.NA,
        "last_trade_date": pd.NaT,
        "strike": np.nan,
        "last_price": np.nan,
        "bid": np.nan,
        "ask": np.nan,
        "change": np.nan,
        "percentChange": np.nan,
        "volume": np.nan,
        "openInterest": np.nan,
        "vendor_implied_vol": np.nan,
        "vendor_in_the_money": pd.NA,
        "contract_size": pd.NA,
        "currency": pd.NA,
        "underlying_spot_snapshot": np.nan,
    }

    for column, default_value in required_columns_with_defaults.items():
        if column not in df.columns:
            df[column] = default_value

    # Stable naming for yfinance-style fields kept from the raw payload.
    if "percentChange" in df.columns:
        df = df.rename(columns={"percentChange": "percent_change"})

    if "openInterest" in df.columns:
        df = df.rename(columns={"openInterest": "open_interest"})

    # Dtype normalization.
    df["snapshot_ts_utc"] = pd.to_datetime(df["snapshot_ts_utc"], utc=True, errors="coerce")
    df["last_trade_date"] = pd.to_datetime(df["last_trade_date"], utc=True, errors="coerce")
    df["expiry"] = pd.to_datetime(df["expiry"], errors="coerce").dt.date

    df["ticker"] = df["ticker"].astype("string").str.upper()
    df["source"] = df["source"].astype("string")
    df["option_type"] = df["option_type"].astype("string").str.lower().str.strip()
    df["contract_symbol"] = df["contract_symbol"].astype("string")
    df["contract_size"] = df["contract_size"].astype("string")
    df["currency"] = df["currency"].astype("string")

    numeric_columns = [
        "strike",
        "last_price",
        "bid",
        "ask",
        "change",
        "percent_change",
        "volume",
        "open_interest",
        "vendor_implied_vol",
        "underlying_spot_snapshot",
    ]

    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    # Calendar maturity features.
    valuation_date = valuation_ts.date()

    df["expiry_ts"] = pd.to_datetime(df["expiry"], errors="coerce")
    df["dte_calendar"] = (df["expiry_ts"].dt.date - valuation_date).apply(
        lambda x: x.days if pd.notna(x) else np.nan
    )
    df["tau_years"] = df["dte_calendar"] / float(config.year_basis)

    # Basic traceability and ordering.
    df["snapshot_date_utc"] = pd.to_datetime(df["snapshot_ts_utc"], utc=True).dt.date
    df["row_id"] = np.arange(len(df), dtype=int)

    # Keep a stable, reader-facing column order.
    ordered_columns = [
        "row_id",
        "snapshot_ts_utc",
        "snapshot_date_utc",
        "ticker",
        "source",
        "option_type",
        "expiry",
        "expiry_ts",
        "dte_calendar",
        "tau_years",
        "strike",
        "contract_symbol",
        "last_trade_date",
        "last_price",
        "bid",
        "ask",
        "change",
        "percent_change",
        "volume",
        "open_interest",
        "vendor_implied_vol",
        "vendor_in_the_money",
        "contract_size",
        "currency",
        "underlying_spot_snapshot",
    ]

    existing_ordered_columns = [column for column in ordered_columns if column in df.columns]
    remaining_columns = [column for column in df.columns if column not in existing_ordered_columns]

    df = df[existing_ordered_columns + remaining_columns]

    return df


def make_integrity_record(
    check_name: str,
    fail_mask: pd.Series,
    severity: str,
    description: str,
) -> dict[str, Any]:
    """
    Convert a boolean failure mask into one integrity-ledger row.
    """
    fail_count = int(fail_mask.fillna(True).sum())

    return {
        "check_name": check_name,
        "status": "PASS" if fail_count == 0 else "FAIL",
        "fail_count": fail_count,
        "severity": severity,
        "description": description,
    }


def build_raw_integrity_ledger(df: pd.DataFrame) -> pd.DataFrame:
    """
    Run hard structural checks on the standardized raw panel.

    These checks do not filter the data yet.
    They only record whether the raw standardized panel has structural problems.
    """
    ledger_records = []

    duplicate_contract_mask = df["contract_symbol"].duplicated(keep=False) & df["contract_symbol"].notna()

    ledger_records.append(
        make_integrity_record(
            check_name="contract_symbol_not_missing",
            fail_mask=df["contract_symbol"].isna() | (df["contract_symbol"].astype("string").str.len() == 0),
            severity="HIGH",
            description="Every row should retain a traceable option contract symbol.",
        )
    )

    ledger_records.append(
        make_integrity_record(
            check_name="no_duplicate_contract_symbols",
            fail_mask=duplicate_contract_mask,
            severity="HIGH",
            description="Each contract symbol should appear only once in the raw snapshot.",
        )
    )

    ledger_records.append(
        make_integrity_record(
            check_name="valid_option_type",
            fail_mask=~df["option_type"].isin(["call", "put"]),
            severity="HIGH",
            description="Option type must be either call or put.",
        )
    )

    ledger_records.append(
        make_integrity_record(
            check_name="expiry_not_missing",
            fail_mask=df["expiry_ts"].isna(),
            severity="HIGH",
            description="Every option quote must have a parseable expiry date.",
        )
    )

    ledger_records.append(
        make_integrity_record(
            check_name="positive_time_to_expiry",
            fail_mask=~(df["tau_years"] > 0),
            severity="HIGH",
            description="Every selected contract should have positive calendar time to expiry.",
        )
    )

    ledger_records.append(
        make_integrity_record(
            check_name="positive_strike",
            fail_mask=~(df["strike"] > 0),
            severity="HIGH",
            description="Every option quote should have a strictly positive strike.",
        )
    )

    ledger_records.append(
        make_integrity_record(
            check_name="bid_not_missing",
            fail_mask=df["bid"].isna(),
            severity="HIGH",
            description="Bid quote should be present before quote-quality filtering.",
        )
    )

    ledger_records.append(
        make_integrity_record(
            check_name="ask_not_missing",
            fail_mask=df["ask"].isna(),
            severity="HIGH",
            description="Ask quote should be present before quote-quality filtering.",
        )
    )

    ledger_records.append(
        make_integrity_record(
            check_name="nonnegative_bid",
            fail_mask=~(df["bid"] >= 0),
            severity="HIGH",
            description="Bid quote must be nonnegative.",
        )
    )

    ledger_records.append(
        make_integrity_record(
            check_name="nonnegative_ask",
            fail_mask=~(df["ask"] >= 0),
            severity="HIGH",
            description="Ask quote must be nonnegative.",
        )
    )

    ledger_records.append(
        make_integrity_record(
            check_name="not_crossed_market",
            fail_mask=df["ask"] < df["bid"],
            severity="HIGH",
            description="Ask should not be below bid.",
        )
    )

    ledger_records.append(
        make_integrity_record(
            check_name="underlying_spot_positive",
            fail_mask=~(df["underlying_spot_snapshot"] > 0),
            severity="HIGH",
            description="Underlying spot snapshot should be strictly positive.",
        )
    )

    integrity_ledger = pd.DataFrame(ledger_records)

    severity_rank = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
    integrity_ledger["severity_rank"] = integrity_ledger["severity"].map(severity_rank).fillna(99)
    integrity_ledger = integrity_ledger.sort_values(
        ["status", "severity_rank", "check_name"],
        ascending=[True, True, True],
    ).drop(columns=["severity_rank"]).reset_index(drop=True)

    return integrity_ledger


standardized_raw_panel = standardize_raw_option_panel(
    raw_df=raw_option_snapshot,
    config=CONFIG,
    valuation_ts=RUN_TIMESTAMP_UTC,
)

raw_integrity_ledger = build_raw_integrity_ledger(standardized_raw_panel)

actual_standardized_panel_path = safe_write_dataframe(
    standardized_raw_panel,
    ARTIFACT_PATHS["raw_standardized_panel"],
)

ARTIFACT_PATHS["raw_standardized_panel"] = actual_standardized_panel_path
MANIFEST["artifact_paths"]["raw_standardized_panel"] = str(actual_standardized_panel_path)

MANIFEST["standardized_raw_rows"] = int(len(standardized_raw_panel))
MANIFEST["standardized_raw_columns"] = list(standardized_raw_panel.columns)
MANIFEST["raw_integrity_passed"] = bool((raw_integrity_ledger["status"] == "PASS").all())
MANIFEST["raw_integrity_failed_checks"] = raw_integrity_ledger.loc[
    raw_integrity_ledger["status"] == "FAIL", "check_name"
].tolist()

print("Raw option-chain panel standardized.")
print(f"Standardized rows: {len(standardized_raw_panel):,}")
print(f"Standardized columns: {len(standardized_raw_panel.columns):,}")
print(f"Raw integrity passed: {MANIFEST['raw_integrity_passed']}")
print(f"Failed integrity checks: {len(MANIFEST['raw_integrity_failed_checks'])}")
print(f"Standardized panel path: {actual_standardized_panel_path}")

display(raw_integrity_ledger)
display(
    standardized_raw_panel[
        [
            "row_id",
            "option_type",
            "expiry",
            "dte_calendar",
            "tau_years",
            "strike",
            "bid",
            "ask",
            "last_price",
            "volume",
            "open_interest",
            "vendor_implied_vol",
            "underlying_spot_snapshot",
        ]
    ].head(10)
)

Raw option-chain panel standardized.
Standardized rows: 3,954
Standardized columns: 25
Raw integrity passed: True
Failed integrity checks: 0
Standardized panel path: d:\Derivative Pricing Project v1.0+\V1.1\data\processed\spy_option_quote_panel_raw_20260705_154512_UTC.parquet


,check_name,status,fail_count,severity,description
0,ask_not_missing,PASS,0,HIGH,Ask quote should be present before quote-quali...
1,bid_not_missing,PASS,0,HIGH,Bid quote should be present before quote-quali...
2,contract_symbol_not_missing,PASS,0,HIGH,Every row should retain a traceable option con...
3,expiry_not_missing,PASS,0,HIGH,Every option quote must have a parseable expir...
4,no_duplicate_contract_symbols,PASS,0,HIGH,Each contract symbol should appear only once i...
5,nonnegative_ask,PASS,0,HIGH,Ask quote must be nonnegative.
6,nonnegative_bid,PASS,0,HIGH,Bid quote must be nonnegative.
7,not_crossed_market,PASS,0,HIGH,Ask should not be below bid.
8,positive_strike,PASS,0,HIGH,Every option quote should have a strictly posi...
9,positive_time_to_expiry,PASS,0,HIGH,Every selected contract should have positive c...


,row_id,option_type,expiry,dte_calendar,tau_years,strike,bid,ask,last_price,volume,open_interest,vendor_implied_vol,underlying_spot_snapshot
0,0,call,2026-07-06,1,0.002740,625.000000,117.820000,121.340000,123.710000,30.000000,31.000000,1.448001,744.780029
1,1,call,2026-07-06,1,0.002740,660.000000,82.830000,86.350000,87.600000,6.000000,26.000000,1.075688,744.780029
2,2,call,2026-07-06,1,0.002740,665.000000,77.830000,81.350000,80.060000,2.000000,0.000000,1.022222,744.780029
3,3,call,2026-07-06,1,0.002740,670.000000,72.830000,76.350000,75.090000,2.000000,12.000000,0.968994,744.780029
4,4,call,2026-07-06,1,0.002740,700.000000,42.840000,46.360000,45.140000,122.000000,5.000000,0.645755,744.780029
5,5,call,2026-07-06,1,0.002740,703.000000,39.840000,43.360000,45.680000,8.000000,8.000000,0.612553,744.780029
6,6,call,2026-07-06,1,0.002740,705.000000,37.850000,41.360000,36.970000,1.000000,16.000000,0.590092,744.780029
7,7,call,2026-07-06,1,0.002740,706.000000,36.890000,40.360000,35.340000,29.000000,31.000000,0.578984,744.780029
8,8,call,2026-07-06,1,0.002740,709.000000,33.850000,37.360000,35.640000,2.000000,4.000000,0.545171,744.780029
9,9,call,2026-07-06,1,0.002740,710.000000,32.850000,36.360000,34.660000,11.000000,23.000000,0.533940,744.780029


In [4]:
# ============================================================
# Cell 5: Quote-quality feature engineering
# ============================================================

def add_quote_quality_features(
    df: pd.DataFrame,
    config: Notebook08Config,
    valuation_ts: datetime,
) -> pd.DataFrame:
    """
    Add quote-quality features to the standardized raw option panel.

    This cell still does not reject rows.
    It only creates features that later rejection logic can use transparently.
    """
    out = df.copy()

    # --------------------------------------------------------
    # Core quote geometry
    # --------------------------------------------------------

    out["mid"] = (out["bid"] + out["ask"]) / 2.0
    out["spread"] = out["ask"] - out["bid"]

    out["rel_spread_mid"] = np.where(
        out["mid"] > config.quote_tol,
        out["spread"] / out["mid"],
        np.nan,
    )

    out["spread_bps_mid"] = 10_000.0 * out["rel_spread_mid"]

    # --------------------------------------------------------
    # Basic quote flags
    # --------------------------------------------------------

    out["has_bid"] = out["bid"].notna() & (out["bid"] > config.quote_tol)
    out["has_ask"] = out["ask"].notna() & (out["ask"] > config.quote_tol)
    out["has_last_price"] = out["last_price"].notna() & (out["last_price"] > config.quote_tol)

    out["is_crossed_market"] = out["ask"] < out["bid"]
    out["is_locked_market"] = np.isclose(out["ask"], out["bid"], atol=config.quote_tol, rtol=0.0)

    out["is_zero_bid"] = out["bid"].notna() & (out["bid"] <= config.quote_tol)
    out["is_zero_ask"] = out["ask"].notna() & (out["ask"] <= config.quote_tol)
    out["is_nonpositive_mid"] = out["mid"].isna() | (out["mid"] <= config.quote_tol)

    out["is_dust_mid"] = out["mid"].notna() & (out["mid"] < config.min_mid_price)
    out["is_wide_abs_spread"] = out["spread"].notna() & (out["spread"] > config.max_abs_spread)
    out["is_wide_rel_spread"] = out["rel_spread_mid"].notna() & (
        out["rel_spread_mid"] > config.max_rel_spread_mid
    )

    out["has_open_interest"] = out["open_interest"].fillna(0) > 0
    out["has_volume"] = out["volume"].fillna(0) > 0

    # --------------------------------------------------------
    # Spot-moneyness features
    # These are only descriptive for Notebook 08.
    # Forward moneyness belongs after parity/forward diagnostics.
    # --------------------------------------------------------

    out["moneyness_spot"] = out["strike"] / out["underlying_spot_snapshot"]

    out["log_moneyness_spot"] = np.where(
        (out["strike"] > 0) & (out["underlying_spot_snapshot"] > 0),
        np.log(out["strike"] / out["underlying_spot_snapshot"]),
        np.nan,
    )

    out["abs_log_moneyness_spot"] = out["log_moneyness_spot"].abs()

    # --------------------------------------------------------
    # Crude spot-intrinsic and extrinsic diagnostics.
    # These are not theoretical pricing bounds yet because rates,
    # dividends, and American exercise are not being modeled here.
    # --------------------------------------------------------

    call_intrinsic_spot = np.maximum(out["underlying_spot_snapshot"] - out["strike"], 0.0)
    put_intrinsic_spot = np.maximum(out["strike"] - out["underlying_spot_snapshot"], 0.0)

    out["intrinsic_spot"] = np.where(
        out["option_type"] == "call",
        call_intrinsic_spot,
        np.where(out["option_type"] == "put", put_intrinsic_spot, np.nan),
    )

    out["extrinsic_mid_vs_spot"] = out["mid"] - out["intrinsic_spot"]

    out["negative_extrinsic_vs_spot"] = (
        out["extrinsic_mid_vs_spot"].notna()
        & (out["extrinsic_mid_vs_spot"] < -config.parity_soft_tol_abs)
    )

    # --------------------------------------------------------
    # Last-trade age diagnostics.
    # This is a stale-quote candidate flag only; quote timestamp
    # conventions differ across vendors.
    # --------------------------------------------------------

    out["last_trade_age_days"] = (
        pd.to_datetime(valuation_ts, utc=True) - pd.to_datetime(out["last_trade_date"], utc=True)
    ).dt.total_seconds() / 86_400.0

    out["missing_last_trade_date"] = out["last_trade_date"].isna()
    out["last_trade_after_snapshot"] = out["last_trade_age_days"] < -config.quote_tol
    out["stale_trade_candidate"] = out["last_trade_age_days"] > 7.0

    # --------------------------------------------------------
    # Descriptive quality bucket.
    # This is not the final rejection reason. It is only a quick
    # reader-facing classification.
    # --------------------------------------------------------

    bucket = np.full(len(out), "clean_candidate", dtype=object)

    bucket[out["bid"].isna() | out["ask"].isna()] = "missing_bid_or_ask"
    bucket[out["is_crossed_market"]] = "crossed_market"
    bucket[out["is_zero_ask"]] = "zero_ask"
    bucket[out["is_nonpositive_mid"]] = "nonpositive_mid"
    bucket[out["is_dust_mid"]] = "dust_mid"
    bucket[out["is_wide_rel_spread"]] = "wide_relative_spread"
    bucket[out["is_wide_abs_spread"]] = "wide_absolute_spread"
    bucket[out["is_zero_bid"] & (bucket == "clean_candidate")] = "zero_bid"
    bucket[out["is_locked_market"] & (bucket == "clean_candidate")] = "locked_market"
    bucket[out["negative_extrinsic_vs_spot"] & (bucket == "clean_candidate")] = "negative_extrinsic_vs_spot"
    bucket[out["stale_trade_candidate"] & (bucket == "clean_candidate")] = "stale_trade_candidate"

    out["quote_quality_bucket"] = bucket

    return out


def summarize_quote_quality(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build compact quote-quality summaries for review.
    """
    bucket_summary = (
        df.groupby(["quote_quality_bucket", "option_type"], dropna=False)
        .size()
        .rename("row_count")
        .reset_index()
        .sort_values(["quote_quality_bucket", "option_type"])
        .reset_index(drop=True)
    )

    expiry_quality_summary = (
        df.groupby(["expiry", "dte_calendar"], dropna=False)
        .agg(
            rows=("row_id", "count"),
            call_rows=("option_type", lambda x: int((x == "call").sum())),
            put_rows=("option_type", lambda x: int((x == "put").sum())),
            median_mid=("mid", "median"),
            median_spread=("spread", "median"),
            median_rel_spread_mid=("rel_spread_mid", "median"),
            zero_bid_rows=("is_zero_bid", "sum"),
            wide_rel_spread_rows=("is_wide_rel_spread", "sum"),
            wide_abs_spread_rows=("is_wide_abs_spread", "sum"),
            dust_mid_rows=("is_dust_mid", "sum"),
            stale_trade_candidate_rows=("stale_trade_candidate", "sum"),
            clean_candidate_bucket_rows=("quote_quality_bucket", lambda x: int((x == "clean_candidate").sum())),
        )
        .reset_index()
        .sort_values(["dte_calendar", "expiry"])
        .reset_index(drop=True)
    )

    option_type_summary = (
        df.groupby("option_type", dropna=False)
        .agg(
            rows=("row_id", "count"),
            median_mid=("mid", "median"),
            median_spread=("spread", "median"),
            median_rel_spread_mid=("rel_spread_mid", "median"),
            median_abs_log_moneyness_spot=("abs_log_moneyness_spot", "median"),
            zero_bid_rows=("is_zero_bid", "sum"),
            zero_ask_rows=("is_zero_ask", "sum"),
            wide_rel_spread_rows=("is_wide_rel_spread", "sum"),
            wide_abs_spread_rows=("is_wide_abs_spread", "sum"),
            dust_mid_rows=("is_dust_mid", "sum"),
            negative_extrinsic_vs_spot_rows=("negative_extrinsic_vs_spot", "sum"),
            stale_trade_candidate_rows=("stale_trade_candidate", "sum"),
        )
        .reset_index()
        .sort_values("option_type")
        .reset_index(drop=True)
    )

    return bucket_summary, expiry_quality_summary, option_type_summary


quote_feature_panel = add_quote_quality_features(
    df=standardized_raw_panel,
    config=CONFIG,
    valuation_ts=RUN_TIMESTAMP_UTC,
)

quote_quality_bucket_summary, expiry_quote_quality_summary, option_type_quote_quality_summary = (
    summarize_quote_quality(quote_feature_panel)
)

MANIFEST["quote_feature_rows"] = int(len(quote_feature_panel))
MANIFEST["quote_quality_bucket_counts"] = (
    quote_feature_panel["quote_quality_bucket"].value_counts(dropna=False).to_dict()
)
MANIFEST["quote_feature_columns_added"] = [
    column for column in quote_feature_panel.columns if column not in standardized_raw_panel.columns
]

print("Quote-quality features added.")
print(f"Rows: {len(quote_feature_panel):,}")
print(f"Feature columns added: {len(MANIFEST['quote_feature_columns_added'])}")
print("Quote-quality bucket counts:")
display(quote_quality_bucket_summary)

print("Option-type quote-quality summary:")
display(option_type_quote_quality_summary)

print("Expiry-level quote-quality summary:")
display(expiry_quote_quality_summary)

display(
    quote_feature_panel[
        [
            "row_id",
            "option_type",
            "expiry",
            "dte_calendar",
            "strike",
            "moneyness_spot",
            "bid",
            "ask",
            "mid",
            "spread",
            "rel_spread_mid",
            "volume",
            "open_interest",
            "last_trade_age_days",
            "quote_quality_bucket",
        ]
    ].head(15)
)

Quote-quality features added.
Rows: 3,954
Feature columns added: 28
Quote-quality bucket counts:


,quote_quality_bucket,option_type,row_count
0,clean_candidate,call,1264
1,clean_candidate,put,1666
2,dust_mid,call,5
3,negative_extrinsic_vs_spot,call,29
4,negative_extrinsic_vs_spot,put,4
5,stale_trade_candidate,call,451
6,stale_trade_candidate,put,113
7,wide_relative_spread,call,260
8,wide_relative_spread,put,162


Option-type quote-quality summary:


,option_type,rows,median_mid,median_spread,median_rel_spread_mid,median_abs_log_moneyness_spot,zero_bid_rows,zero_ask_rows,wide_rel_spread_rows,wide_abs_spread_rows,dust_mid_rows,negative_extrinsic_vs_spot_rows,stale_trade_candidate_rows
0,call,2009,19.140000,0.770000,0.041343,0.061472,148,5,260,0,142,34,543
1,put,1945,1.865000,0.020000,0.032258,0.070617,101,0,162,0,101,4,141


Expiry-level quote-quality summary:


,expiry,dte_calendar,rows,call_rows,put_rows,median_mid,median_spread,median_rel_spread_mid,zero_bid_rows,wide_rel_spread_rows,wide_abs_spread_rows,dust_mid_rows,stale_trade_candidate_rows,clean_candidate_bucket_rows
0,2026-07-06,1,201,99,102,0.025000,0.010000,0.400000,73,95,0,73,13,98
1,2026-07-07,2,174,87,87,0.075000,0.010000,0.181818,45,60,0,45,12,111
2,2026-07-08,3,170,83,87,0.165000,0.010000,0.104656,27,51,0,27,25,110
3,2026-07-09,4,146,71,75,0.352500,0.010000,0.069357,12,29,0,12,7,113
4,2026-07-10,5,374,192,182,1.045000,0.020000,0.076741,38,78,0,38,59,247
5,2026-07-17,12,450,241,209,3.020000,0.030000,0.051513,35,59,0,35,100,313
6,2026-07-24,19,331,169,162,3.255000,0.040000,0.044444,7,15,0,6,48,272
7,2026-07-31,26,469,242,227,7.565000,0.050000,0.029838,4,11,0,2,101,361
8,2026-08-07,33,197,94,103,4.700000,0.040000,0.020000,0,2,0,0,9,187
9,2026-08-21,47,447,238,209,10.700000,0.060000,0.021505,5,20,0,2,74,360


,row_id,option_type,expiry,dte_calendar,strike,moneyness_spot,bid,ask,mid,spread,rel_spread_mid,volume,open_interest,last_trade_age_days,quote_quality_bucket
0,0,call,2026-07-06,1,625.000000,0.839174,117.820000,121.340000,119.580000,3.520000,0.029436,30.000000,31.000000,3.045521,clean_candidate
1,1,call,2026-07-06,1,660.000000,0.886168,82.830000,86.350000,84.590000,3.520000,0.041612,6.000000,26.000000,6.091099,clean_candidate
2,2,call,2026-07-06,1,665.000000,0.892881,77.830000,81.350000,79.590000,3.520000,0.044227,2.000000,0.000000,2.824815,clean_candidate
3,3,call,2026-07-06,1,670.000000,0.899594,72.830000,76.350000,74.590000,3.520000,0.047191,2.000000,12.000000,4.047349,clean_candidate
4,4,call,2026-07-06,1,700.000000,0.939875,42.840000,46.360000,44.600000,3.520000,0.078924,122.000000,5.000000,2.824861,clean_candidate
5,5,call,2026-07-06,1,703.000000,0.943903,39.840000,43.360000,41.600000,3.520000,0.084615,8.000000,8.000000,4.851898,clean_candidate
6,6,call,2026-07-06,1,705.000000,0.946588,37.850000,41.360000,39.605000,3.510000,0.088625,1.000000,16.000000,2.935393,clean_candidate
7,7,call,2026-07-06,1,706.000000,0.947931,36.890000,40.360000,38.625000,3.470000,0.089838,29.000000,31.000000,5.910000,clean_candidate
8,8,call,2026-07-06,1,709.000000,0.951959,33.850000,37.360000,35.605000,3.510000,0.098582,2.000000,4.000000,2.825949,clean_candidate
9,9,call,2026-07-06,1,710.000000,0.953302,32.850000,36.360000,34.605000,3.510000,0.101430,11.000000,23.000000,2.825960,clean_candidate


In [5]:
# ============================================================
# Cell 6: Quote rejection rules, filtered panel, and rejection ledger
# ============================================================

def add_quote_rejection_flags(
    df: pd.DataFrame,
    config: Notebook08Config,
) -> pd.DataFrame:
    """
    Add explicit quote-rejection flags.

    This cell separates:
        1. hard rejection rules used to build the quote-quality-filtered panel
        2. soft warning flags retained for downstream interpretation

    Important:
        Stale last-trade information is treated as a warning, not a hard rejection.
        yfinance lastTradeDate is a trade timestamp, not necessarily the live quote timestamp.
    """
    out = df.copy()

    # --------------------------------------------------------
    # Hard rejection flags
    # --------------------------------------------------------

    out["reject_invalid_option_type"] = ~out["option_type"].isin(["call", "put"])
    out["reject_missing_contract_symbol"] = (
        out["contract_symbol"].isna()
        | (out["contract_symbol"].astype("string").str.len() == 0)
    )

    out["reject_missing_expiry"] = out["expiry_ts"].isna()
    out["reject_nonpositive_tau"] = out["tau_years"].isna() | (out["tau_years"] <= 0)
    out["reject_too_short_maturity"] = out["dte_calendar"].isna() | (
        out["dte_calendar"] < config.min_dte_calendar
    )
    out["reject_too_long_maturity"] = out["dte_calendar"].isna() | (
        out["dte_calendar"] > config.max_dte_calendar
    )

    out["reject_missing_or_nonpositive_strike"] = out["strike"].isna() | (out["strike"] <= 0)

    out["reject_missing_bid"] = out["bid"].isna()
    out["reject_missing_ask"] = out["ask"].isna()
    out["reject_negative_bid"] = out["bid"].notna() & (out["bid"] < -config.quote_tol)
    out["reject_negative_ask"] = out["ask"].notna() & (out["ask"] < -config.quote_tol)
    out["reject_crossed_market"] = out["ask"] < out["bid"]

    out["reject_nonpositive_mid"] = out["mid"].isna() | (out["mid"] <= config.quote_tol)
    out["reject_dust_mid"] = out["mid"].notna() & (out["mid"] < config.min_mid_price)

    out["reject_missing_spread"] = out["spread"].isna()
    out["reject_negative_spread"] = out["spread"].notna() & (out["spread"] < -config.quote_tol)
    out["reject_too_wide_abs_spread"] = out["spread"].notna() & (
        out["spread"] > config.max_abs_spread
    )
    out["reject_too_wide_rel_spread"] = out["rel_spread_mid"].notna() & (
        out["rel_spread_mid"] > config.max_rel_spread_mid
    )

    out["reject_low_open_interest"] = (
        out["open_interest"].fillna(0) < config.min_open_interest
    )
    out["reject_low_volume"] = out["volume"].fillna(0) < config.min_volume

    out["reject_missing_underlying_spot"] = (
        out["underlying_spot_snapshot"].isna()
        | (out["underlying_spot_snapshot"] <= 0)
    )

    # --------------------------------------------------------
    # Soft warning flags
    # --------------------------------------------------------

    out["warn_zero_bid"] = out["is_zero_bid"]
    out["warn_locked_market"] = out["is_locked_market"]
    out["warn_missing_last_trade_date"] = out["missing_last_trade_date"]
    out["warn_last_trade_after_snapshot"] = out["last_trade_after_snapshot"]
    out["warn_stale_trade_candidate"] = out["stale_trade_candidate"]

    # This is only a spot-based sanity flag.
    # It is not treated as a formal arbitrage rejection.
    out["warn_negative_extrinsic_vs_spot"] = out["negative_extrinsic_vs_spot"]

    # --------------------------------------------------------
    # Aggregate hard rejection status
    # --------------------------------------------------------

    hard_rejection_columns = [
        column for column in out.columns if column.startswith("reject_")
    ]

    warning_columns = [
        column for column in out.columns if column.startswith("warn_")
    ]

    out[hard_rejection_columns] = out[hard_rejection_columns].fillna(False).astype(bool)
    out[warning_columns] = out[warning_columns].fillna(False).astype(bool)

    out["rejection_flag_count"] = out[hard_rejection_columns].sum(axis=1).astype(int)
    out["warning_flag_count"] = out[warning_columns].sum(axis=1).astype(int)

    out["is_quote_quality_pass"] = out["rejection_flag_count"] == 0

    # --------------------------------------------------------
    # Primary rejection reason
    # Ordered from structural errors to quote-quality errors.
    # --------------------------------------------------------

    rejection_priority = [
        "reject_invalid_option_type",
        "reject_missing_contract_symbol",
        "reject_missing_expiry",
        "reject_nonpositive_tau",
        "reject_too_short_maturity",
        "reject_too_long_maturity",
        "reject_missing_or_nonpositive_strike",
        "reject_missing_bid",
        "reject_missing_ask",
        "reject_negative_bid",
        "reject_negative_ask",
        "reject_crossed_market",
        "reject_missing_underlying_spot",
        "reject_nonpositive_mid",
        "reject_dust_mid",
        "reject_missing_spread",
        "reject_negative_spread",
        "reject_too_wide_abs_spread",
        "reject_too_wide_rel_spread",
        "reject_low_open_interest",
        "reject_low_volume",
    ]

    primary_reason = np.full(len(out), "PASS", dtype=object)

    for reason in rejection_priority:
        if reason in out.columns:
            mask = (primary_reason == "PASS") & out[reason]
            primary_reason[mask] = reason

    out["primary_rejection_reason"] = primary_reason

    return out


def build_rejection_ledger(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build row-level rejection ledger.

    Each row in this ledger corresponds to one rejected option quote.
    """
    rejection_columns = [
        column for column in df.columns if column.startswith("reject_")
    ]

    warning_columns = [
        column for column in df.columns if column.startswith("warn_")
    ]

    rejected = df.loc[~df["is_quote_quality_pass"]].copy()

    ledger_columns = [
        "row_id",
        "contract_symbol",
        "option_type",
        "expiry",
        "dte_calendar",
        "tau_years",
        "strike",
        "bid",
        "ask",
        "mid",
        "spread",
        "rel_spread_mid",
        "volume",
        "open_interest",
        "vendor_implied_vol",
        "last_trade_date",
        "last_trade_age_days",
        "quote_quality_bucket",
        "primary_rejection_reason",
        "rejection_flag_count",
        "warning_flag_count",
    ]

    existing_columns = [
        column for column in ledger_columns + rejection_columns + warning_columns
        if column in rejected.columns
    ]

    ledger = rejected[existing_columns].sort_values(
        ["primary_rejection_reason", "expiry", "option_type", "strike"]
    ).reset_index(drop=True)

    return ledger


def summarize_rejections(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build rejection summaries by reason, option type, and expiry.
    """
    rejection_reason_summary = (
        df.groupby(["primary_rejection_reason", "option_type"], dropna=False)
        .size()
        .rename("row_count")
        .reset_index()
        .sort_values(["primary_rejection_reason", "option_type"])
        .reset_index(drop=True)
    )

    rejection_flag_columns = [
        column for column in df.columns if column.startswith("reject_")
    ]

    rejection_flag_summary = (
        df[rejection_flag_columns]
        .sum()
        .rename("row_count")
        .reset_index()
        .rename(columns={"index": "rejection_flag"})
        .sort_values("row_count", ascending=False)
        .reset_index(drop=True)
    )

    expiry_rejection_summary = (
        df.groupby(["expiry", "dte_calendar"], dropna=False)
        .agg(
            raw_rows=("row_id", "count"),
            passed_rows=("is_quote_quality_pass", "sum"),
            rejected_rows=("is_quote_quality_pass", lambda x: int((~x).sum())),
            median_mid=("mid", "median"),
            median_spread=("spread", "median"),
            median_rel_spread_mid=("rel_spread_mid", "median"),
            wide_rel_spread_rows=("reject_too_wide_rel_spread", "sum"),
            dust_mid_rows=("reject_dust_mid", "sum"),
            crossed_market_rows=("reject_crossed_market", "sum"),
            soft_warning_rows=("warning_flag_count", lambda x: int((x > 0).sum())),
        )
        .reset_index()
        .sort_values(["dte_calendar", "expiry"])
        .reset_index(drop=True)
    )

    expiry_rejection_summary["pass_rate"] = (
        expiry_rejection_summary["passed_rows"] / expiry_rejection_summary["raw_rows"]
    )

    return rejection_reason_summary, rejection_flag_summary, expiry_rejection_summary


quote_screened_panel = add_quote_rejection_flags(
    df=quote_feature_panel,
    config=CONFIG,
)

quote_filtered_panel = (
    quote_screened_panel.loc[quote_screened_panel["is_quote_quality_pass"]]
    .copy()
    .sort_values(["expiry", "option_type", "strike"])
    .reset_index(drop=True)
)

quote_rejection_ledger = build_rejection_ledger(quote_screened_panel)

rejection_reason_summary, rejection_flag_summary, expiry_rejection_summary = summarize_rejections(
    quote_screened_panel
)

actual_filtered_panel_path = safe_write_dataframe(
    quote_filtered_panel,
    ARTIFACT_PATHS["quote_filtered_panel"],
)

ARTIFACT_PATHS["quote_filtered_panel"] = actual_filtered_panel_path
MANIFEST["artifact_paths"]["quote_filtered_panel"] = str(actual_filtered_panel_path)

actual_rejection_ledger_path = ARTIFACT_PATHS["rejection_ledger"]
actual_rejection_ledger_path.parent.mkdir(parents=True, exist_ok=True)
quote_rejection_ledger.to_csv(actual_rejection_ledger_path, index=False)

MANIFEST["artifact_paths"]["rejection_ledger"] = str(actual_rejection_ledger_path)

MANIFEST["quote_screened_rows"] = int(len(quote_screened_panel))
MANIFEST["quote_filtered_rows"] = int(len(quote_filtered_panel))
MANIFEST["quote_rejected_rows"] = int(len(quote_rejection_ledger))
MANIFEST["quote_filter_pass_rate"] = float(len(quote_filtered_panel) / len(quote_screened_panel))
MANIFEST["primary_rejection_reason_counts"] = (
    quote_screened_panel["primary_rejection_reason"].value_counts(dropna=False).to_dict()
)
MANIFEST["hard_rejection_columns"] = [
    column for column in quote_screened_panel.columns if column.startswith("reject_")
]
MANIFEST["soft_warning_columns"] = [
    column for column in quote_screened_panel.columns if column.startswith("warn_")
]

print("Quote rejection rules applied.")
print(f"Raw screened rows: {len(quote_screened_panel):,}")
print(f"Quote-quality passed rows: {len(quote_filtered_panel):,}")
print(f"Rejected rows: {len(quote_rejection_ledger):,}")
print(f"Pass rate: {MANIFEST['quote_filter_pass_rate']:.2%}")
print(f"Filtered panel path: {actual_filtered_panel_path}")
print(f"Rejection ledger path: {actual_rejection_ledger_path}")

print("Primary rejection reason summary:")
display(rejection_reason_summary)

print("Rejection flag summary:")
display(rejection_flag_summary)

print("Expiry rejection summary:")
display(expiry_rejection_summary)

print("Filtered panel preview:")
display(
    quote_filtered_panel[
        [
            "row_id",
            "option_type",
            "expiry",
            "dte_calendar",
            "strike",
            "moneyness_spot",
            "bid",
            "ask",
            "mid",
            "spread",
            "rel_spread_mid",
            "volume",
            "open_interest",
            "warning_flag_count",
            "quote_quality_bucket",
            "primary_rejection_reason",
        ]
    ].head(20)
)

Quote rejection rules applied.
Raw screened rows: 3,954
Quote-quality passed rows: 3,527
Rejected rows: 427
Pass rate: 89.20%
Filtered panel path: d:\Derivative Pricing Project v1.0+\V1.1\data\processed\spy_option_quote_panel_filtered_20260705_154512_UTC.parquet
Rejection ledger path: d:\Derivative Pricing Project v1.0+\V1.1\data\processed\spy_option_rejection_ledger_20260705_154512_UTC.csv
Primary rejection reason summary:


,primary_rejection_reason,option_type,row_count
0,PASS,call,1744
1,PASS,put,1783
2,reject_dust_mid,call,137
3,reject_dust_mid,put,101
4,reject_nonpositive_mid,call,5
5,reject_too_wide_rel_spread,call,123
6,reject_too_wide_rel_spread,put,61


Rejection flag summary:


,rejection_flag,row_count
0,reject_too_wide_rel_spread,422
1,reject_dust_mid,243
2,reject_nonpositive_mid,5
3,reject_missing_contract_symbol,0
4,reject_invalid_option_type,0
5,reject_too_short_maturity,0
6,reject_nonpositive_tau,0
7,reject_missing_expiry,0
8,reject_too_long_maturity,0
9,reject_missing_ask,0


Expiry rejection summary:


,expiry,dte_calendar,raw_rows,passed_rows,rejected_rows,median_mid,median_spread,median_rel_spread_mid,wide_rel_spread_rows,dust_mid_rows,crossed_market_rows,soft_warning_rows,pass_rate
0,2026-07-06,1,201,106,95,0.025000,0.010000,0.400000,95,73,0,81,0.527363
1,2026-07-07,2,174,114,60,0.075000,0.010000,0.181818,60,45,0,48,0.655172
2,2026-07-08,3,170,119,51,0.165000,0.010000,0.104656,51,27,0,37,0.700000
3,2026-07-09,4,146,117,29,0.352500,0.010000,0.069357,29,12,0,16,0.801370
4,2026-07-10,5,374,296,78,1.045000,0.020000,0.076741,78,38,0,87,0.791444
5,2026-07-17,12,450,391,59,3.020000,0.030000,0.051513,59,35,0,113,0.868889
6,2026-07-24,19,331,316,15,3.255000,0.040000,0.044444,15,6,0,55,0.954683
7,2026-07-31,26,469,456,13,7.565000,0.050000,0.029838,11,2,0,103,0.972281
8,2026-08-07,33,197,195,2,4.700000,0.040000,0.020000,2,0,0,9,0.989848
9,2026-08-21,47,447,427,20,10.700000,0.060000,0.021505,20,2,0,76,0.955257


Filtered panel preview:


,row_id,option_type,expiry,dte_calendar,strike,moneyness_spot,bid,ask,mid,spread,rel_spread_mid,volume,open_interest,warning_flag_count,quote_quality_bucket,primary_rejection_reason
0,0,call,2026-07-06,1,625.000000,0.839174,117.820000,121.340000,119.580000,3.520000,0.029436,30.000000,31.000000,0,clean_candidate,PASS
1,1,call,2026-07-06,1,660.000000,0.886168,82.830000,86.350000,84.590000,3.520000,0.041612,6.000000,26.000000,0,clean_candidate,PASS
2,2,call,2026-07-06,1,665.000000,0.892881,77.830000,81.350000,79.590000,3.520000,0.044227,2.000000,0.000000,0,clean_candidate,PASS
3,3,call,2026-07-06,1,670.000000,0.899594,72.830000,76.350000,74.590000,3.520000,0.047191,2.000000,12.000000,0,clean_candidate,PASS
4,4,call,2026-07-06,1,700.000000,0.939875,42.840000,46.360000,44.600000,3.520000,0.078924,122.000000,5.000000,0,clean_candidate,PASS
5,5,call,2026-07-06,1,703.000000,0.943903,39.840000,43.360000,41.600000,3.520000,0.084615,8.000000,8.000000,0,clean_candidate,PASS
6,6,call,2026-07-06,1,705.000000,0.946588,37.850000,41.360000,39.605000,3.510000,0.088625,1.000000,16.000000,0,clean_candidate,PASS
7,7,call,2026-07-06,1,706.000000,0.947931,36.890000,40.360000,38.625000,3.470000,0.089838,29.000000,31.000000,0,clean_candidate,PASS
8,8,call,2026-07-06,1,709.000000,0.951959,33.850000,37.360000,35.605000,3.510000,0.098582,2.000000,4.000000,0,clean_candidate,PASS
9,9,call,2026-07-06,1,710.000000,0.953302,32.850000,36.360000,34.605000,3.510000,0.101430,11.000000,23.000000,0,clean_candidate,PASS


In [6]:
# ============================================================
# Cell 7: Match filtered calls and puts by expiry and strike
# ============================================================

def build_call_put_matched_pairs(
    filtered_df: pd.DataFrame,
    config: Notebook08Config,
) -> pd.DataFrame:
    """
    Build a matched call-put panel from the quote-quality-filtered option panel.

    One row = one expiry-strike pair with both a call and a put available.
    """
    required_columns = [
        "row_id",
        "contract_symbol",
        "expiry",
        "expiry_ts",
        "dte_calendar",
        "tau_years",
        "strike",
        "option_type",
        "bid",
        "ask",
        "mid",
        "spread",
        "rel_spread_mid",
        "volume",
        "open_interest",
        "vendor_implied_vol",
        "vendor_in_the_money",
        "last_trade_date",
        "last_trade_age_days",
        "warning_flag_count",
        "quote_quality_bucket",
        "underlying_spot_snapshot",
        "moneyness_spot",
        "log_moneyness_spot",
    ]

    existing_columns = [column for column in required_columns if column in filtered_df.columns]

    base = filtered_df[existing_columns].copy()

    calls = (
        base.loc[base["option_type"] == "call"]
        .drop(columns=["option_type"])
        .rename(
            columns={
                "row_id": "call_row_id",
                "contract_symbol": "call_contract_symbol",
                "bid": "call_bid",
                "ask": "call_ask",
                "mid": "call_mid",
                "spread": "call_spread",
                "rel_spread_mid": "call_rel_spread_mid",
                "volume": "call_volume",
                "open_interest": "call_open_interest",
                "vendor_implied_vol": "call_vendor_implied_vol",
                "vendor_in_the_money": "call_vendor_in_the_money",
                "last_trade_date": "call_last_trade_date",
                "last_trade_age_days": "call_last_trade_age_days",
                "warning_flag_count": "call_warning_flag_count",
                "quote_quality_bucket": "call_quote_quality_bucket",
            }
        )
    )

    puts = (
        base.loc[base["option_type"] == "put"]
        .drop(columns=["option_type"])
        .rename(
            columns={
                "row_id": "put_row_id",
                "contract_symbol": "put_contract_symbol",
                "bid": "put_bid",
                "ask": "put_ask",
                "mid": "put_mid",
                "spread": "put_spread",
                "rel_spread_mid": "put_rel_spread_mid",
                "volume": "put_volume",
                "open_interest": "put_open_interest",
                "vendor_implied_vol": "put_vendor_implied_vol",
                "vendor_in_the_money": "put_vendor_in_the_money",
                "last_trade_date": "put_last_trade_date",
                "last_trade_age_days": "put_last_trade_age_days",
                "warning_flag_count": "put_warning_flag_count",
                "quote_quality_bucket": "put_quote_quality_bucket",
            }
        )
    )

    join_keys = [
        "expiry",
        "expiry_ts",
        "dte_calendar",
        "tau_years",
        "strike",
        "underlying_spot_snapshot",
        "moneyness_spot",
        "log_moneyness_spot",
    ]

    matched = calls.merge(
        puts,
        on=join_keys,
        how="inner",
        validate="one_to_one",
    )

    matched = matched.sort_values(["expiry", "strike"]).reset_index(drop=True)

    matched["pair_id"] = np.arange(len(matched), dtype=int)

    matched["pair_mid_sum"] = matched["call_mid"] + matched["put_mid"]
    matched["pair_spread_sum"] = matched["call_spread"] + matched["put_spread"]

    matched["pair_max_rel_spread_mid"] = matched[
        ["call_rel_spread_mid", "put_rel_spread_mid"]
    ].max(axis=1)

    matched["pair_mean_rel_spread_mid"] = matched[
        ["call_rel_spread_mid", "put_rel_spread_mid"]
    ].mean(axis=1)

    matched["pair_warning_flag_count"] = (
        matched["call_warning_flag_count"].fillna(0).astype(int)
        + matched["put_warning_flag_count"].fillna(0).astype(int)
    )

    matched["both_sides_have_volume"] = (
        matched["call_volume"].fillna(0) > 0
    ) & (
        matched["put_volume"].fillna(0) > 0
    )

    matched["both_sides_have_open_interest"] = (
        matched["call_open_interest"].fillna(0) > 0
    ) & (
        matched["put_open_interest"].fillna(0) > 0
    )

    matched["pair_quality_pass"] = True

    # Keep pair_id first.
    front_columns = [
        "pair_id",
        "expiry",
        "expiry_ts",
        "dte_calendar",
        "tau_years",
        "strike",
        "underlying_spot_snapshot",
        "moneyness_spot",
        "log_moneyness_spot",
    ]

    remaining_columns = [column for column in matched.columns if column not in front_columns]
    matched = matched[front_columns + remaining_columns]

    return matched


def summarize_matching(
    filtered_df: pd.DataFrame,
    matched_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Summarize call-put matching coverage by expiry.
    """
    filtered_counts = (
        filtered_df.groupby(["expiry", "dte_calendar", "option_type"], dropna=False)
        .size()
        .rename("rows")
        .reset_index()
        .pivot_table(
            index=["expiry", "dte_calendar"],
            columns="option_type",
            values="rows",
            fill_value=0,
        )
        .reset_index()
        .rename(columns={"call": "filtered_call_rows", "put": "filtered_put_rows"})
    )

    for column in ["filtered_call_rows", "filtered_put_rows"]:
        if column not in filtered_counts.columns:
            filtered_counts[column] = 0

    matched_counts = (
        matched_df.groupby(["expiry", "dte_calendar"], dropna=False)
        .agg(
            matched_pairs=("pair_id", "count"),
            min_matched_strike=("strike", "min"),
            max_matched_strike=("strike", "max"),
            median_pair_max_rel_spread_mid=("pair_max_rel_spread_mid", "median"),
            median_pair_mean_rel_spread_mid=("pair_mean_rel_spread_mid", "median"),
            pairs_with_any_warning=("pair_warning_flag_count", lambda x: int((x > 0).sum())),
            pairs_with_both_volume=("both_sides_have_volume", "sum"),
            pairs_with_both_open_interest=("both_sides_have_open_interest", "sum"),
        )
        .reset_index()
    )

    matching_summary = filtered_counts.merge(
        matched_counts,
        on=["expiry", "dte_calendar"],
        how="left",
    )

    fill_zero_columns = [
        "matched_pairs",
        "pairs_with_any_warning",
        "pairs_with_both_volume",
        "pairs_with_both_open_interest",
    ]

    for column in fill_zero_columns:
        matching_summary[column] = matching_summary[column].fillna(0).astype(int)

    matching_summary["call_match_rate"] = np.where(
        matching_summary["filtered_call_rows"] > 0,
        matching_summary["matched_pairs"] / matching_summary["filtered_call_rows"],
        np.nan,
    )

    matching_summary["put_match_rate"] = np.where(
        matching_summary["filtered_put_rows"] > 0,
        matching_summary["matched_pairs"] / matching_summary["filtered_put_rows"],
        np.nan,
    )

    matching_summary["min_side_match_rate"] = matching_summary[
        ["call_match_rate", "put_match_rate"]
    ].min(axis=1)

    matching_summary = matching_summary.sort_values(
        ["dte_calendar", "expiry"]
    ).reset_index(drop=True)

    overall_summary = pd.DataFrame(
        [
            {
                "filtered_rows": int(len(filtered_df)),
                "filtered_call_rows": int((filtered_df["option_type"] == "call").sum()),
                "filtered_put_rows": int((filtered_df["option_type"] == "put").sum()),
                "matched_pairs": int(len(matched_df)),
                "matched_rows_equivalent": int(2 * len(matched_df)),
                "unmatched_filtered_rows": int(len(filtered_df) - 2 * len(matched_df)),
                "expiry_count_filtered": int(filtered_df["expiry"].nunique()),
                "expiry_count_matched": int(matched_df["expiry"].nunique()),
            }
        ]
    )

    overall_summary["matched_row_coverage"] = (
        overall_summary["matched_rows_equivalent"] / overall_summary["filtered_rows"]
    )

    return matching_summary, overall_summary


matched_call_put_pairs = build_call_put_matched_pairs(
    filtered_df=quote_filtered_panel,
    config=CONFIG,
)

matching_expiry_summary, matching_overall_summary = summarize_matching(
    filtered_df=quote_filtered_panel,
    matched_df=matched_call_put_pairs,
)

actual_matched_pairs_path = safe_write_dataframe(
    matched_call_put_pairs,
    ARTIFACT_PATHS["matched_pairs"],
)

ARTIFACT_PATHS["matched_pairs"] = actual_matched_pairs_path
MANIFEST["artifact_paths"]["matched_pairs"] = str(actual_matched_pairs_path)

MANIFEST["matched_pairs"] = int(len(matched_call_put_pairs))
MANIFEST["matched_expiry_count"] = int(matched_call_put_pairs["expiry"].nunique())
MANIFEST["matched_rows_equivalent"] = int(2 * len(matched_call_put_pairs))
MANIFEST["unmatched_filtered_rows"] = int(len(quote_filtered_panel) - 2 * len(matched_call_put_pairs))
MANIFEST["matched_row_coverage"] = float(
    MANIFEST["matched_rows_equivalent"] / len(quote_filtered_panel)
)

print("Call-put matching completed.")
print(f"Filtered rows: {len(quote_filtered_panel):,}")
print(f"Matched call-put pairs: {len(matched_call_put_pairs):,}")
print(f"Matched expiries: {MANIFEST['matched_expiry_count']:,}")
print(f"Matched row coverage: {MANIFEST['matched_row_coverage']:.2%}")
print(f"Unmatched filtered rows: {MANIFEST['unmatched_filtered_rows']:,}")
print(f"Matched pairs path: {actual_matched_pairs_path}")

print("Overall matching summary:")
display(matching_overall_summary)

print("Expiry-level matching summary:")
display(matching_expiry_summary)

print("Matched call-put pair preview:")
display(
    matched_call_put_pairs[
        [
            "pair_id",
            "expiry",
            "dte_calendar",
            "tau_years",
            "strike",
            "moneyness_spot",
            "call_mid",
            "put_mid",
            "call_spread",
            "put_spread",
            "call_rel_spread_mid",
            "put_rel_spread_mid",
            "pair_max_rel_spread_mid",
            "pair_warning_flag_count",
            "both_sides_have_volume",
            "both_sides_have_open_interest",
        ]
    ].head(20)
)

Call-put matching completed.
Filtered rows: 3,527
Matched call-put pairs: 1,510
Matched expiries: 12
Matched row coverage: 85.63%
Unmatched filtered rows: 507
Matched pairs path: d:\Derivative Pricing Project v1.0+\V1.1\data\processed\spy_option_matched_pairs_20260705_154512_UTC.parquet
Overall matching summary:


,filtered_rows,filtered_call_rows,filtered_put_rows,matched_pairs,matched_rows_equivalent,unmatched_filtered_rows,expiry_count_filtered,expiry_count_matched,matched_row_coverage
0,3527,1744,1783,1510,3020,507,12,12,0.856252


Expiry-level matching summary:


,expiry,dte_calendar,filtered_call_rows,filtered_put_rows,matched_pairs,min_matched_strike,max_matched_strike,median_pair_max_rel_spread_mid,median_pair_mean_rel_spread_mid,pairs_with_any_warning,pairs_with_both_volume,pairs_with_both_open_interest,call_match_rate,put_match_rate,min_side_match_rate
0,2026-07-06,1,54.000000,52.000000,42,712.000000,757.000000,0.093495,0.078362,7,41,41,0.777778,0.807692,0.777778
1,2026-07-07,2,50.000000,64.000000,46,690.000000,762.000000,0.061484,0.048777,2,46,44,0.920000,0.718750,0.718750
2,2026-07-08,3,56.000000,63.000000,45,675.000000,760.000000,0.049261,0.036245,8,41,44,0.803571,0.714286,0.714286
3,2026-07-09,4,54.000000,63.000000,37,685.000000,760.000000,0.039260,0.029167,4,34,36,0.685185,0.587302,0.587302
4,2026-07-10,5,147.000000,149.000000,130,615.000000,775.000000,0.103899,0.079229,27,123,121,0.884354,0.872483,0.872483
5,2026-07-17,12,202.000000,189.000000,178,460.000000,800.000000,0.102056,0.059318,56,167,155,0.881188,0.941799,0.881188
6,2026-07-24,19,154.000000,162.000000,130,500.000000,820.000000,0.095885,0.054943,43,117,118,0.844156,0.802469,0.802469
7,2026-07-31,26,229.000000,227.000000,216,375.000000,825.000000,0.070123,0.041011,94,212,197,0.943231,0.951542,0.943231
8,2026-08-07,33,92.000000,103.000000,50,625.000000,805.000000,0.105877,0.059525,8,49,48,0.543478,0.485437,0.485437
9,2026-08-21,47,218.000000,209.000000,202,360.000000,850.000000,0.058807,0.034753,66,194,189,0.926606,0.966507,0.926606


Matched call-put pair preview:


,pair_id,expiry,dte_calendar,tau_years,strike,moneyness_spot,call_mid,put_mid,call_spread,put_spread,call_rel_spread_mid,put_rel_spread_mid,pair_max_rel_spread_mid,pair_warning_flag_count,both_sides_have_volume,both_sides_have_open_interest
0,0,2026-07-06,1,0.002740,712.000000,0.955987,32.610000,0.025000,3.520000,0.010000,0.107942,0.400000,0.400000,0,True,True
1,1,2026-07-06,1,0.002740,713.000000,0.957330,31.610000,0.025000,3.520000,0.010000,0.111357,0.400000,0.400000,1,False,True
2,2,2026-07-06,1,0.002740,714.000000,0.958672,30.610000,0.025000,3.520000,0.010000,0.114995,0.400000,0.400000,0,True,True
3,3,2026-07-06,1,0.002740,715.000000,0.960015,29.610000,0.025000,3.520000,0.010000,0.118879,0.400000,0.400000,0,True,True
4,4,2026-07-06,1,0.002740,717.000000,0.962700,27.615000,0.025000,3.510000,0.010000,0.127105,0.400000,0.400000,0,True,True
5,5,2026-07-06,1,0.002740,720.000000,0.966728,24.780000,0.035000,2.000000,0.010000,0.080710,0.285714,0.285714,0,True,True
6,6,2026-07-06,1,0.002740,721.000000,0.968071,23.625000,0.035000,3.510000,0.010000,0.148571,0.285714,0.285714,0,True,True
7,7,2026-07-06,1,0.002740,722.000000,0.969414,22.630000,0.035000,3.520000,0.010000,0.155546,0.285714,0.285714,0,True,True
8,8,2026-07-06,1,0.002740,723.000000,0.970756,21.385000,0.045000,3.030000,0.010000,0.141688,0.222222,0.222222,1,True,True
9,9,2026-07-06,1,0.002740,724.000000,0.972099,20.435000,0.045000,1.770000,0.010000,0.086616,0.222222,0.222222,1,True,True


In [7]:
# ============================================================
# Cell 8: Put-call parity-implied forward diagnostics
# ============================================================

def median_absolute_deviation(values: pd.Series) -> float:
    """
    Robust median absolute deviation helper.
    Returns NaN for empty or all-missing inputs.
    """
    clean = pd.to_numeric(values, errors="coerce").dropna()

    if clean.empty:
        return np.nan

    median_value = clean.median()
    return float((clean - median_value).abs().median())


def add_parity_forward_diagnostics(
    matched_df: pd.DataFrame,
    config: Notebook08Config,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Add first-pass put-call parity diagnostics to matched call-put pairs.

    This is a quote-quality diagnostic, not a final pricing model.

    In a clean European market with deterministic rates and dividends:

        C - P = D(T) * [F(T) - K]

    Because Notebook 08 has not yet built a discount/dividend curve, this cell uses
    a first-pass undiscounted proxy:

        F_proxy(K,T) = K + C_mid - P_mid

    Bid-ask-aware interval proxy:

        F_lower = K + C_bid - P_ask
        F_upper = K + C_ask - P_bid

    Later notebooks may replace this with an explicit discount/dividend model.
    """
    out = matched_df.copy()

    # --------------------------------------------------------
    # Per-pair parity-implied forward proxy
    # --------------------------------------------------------

    out["parity_forward_mid_proxy"] = (
        out["strike"] + out["call_mid"] - out["put_mid"]
    )

    out["parity_forward_lower_proxy"] = (
        out["strike"] + out["call_bid"] - out["put_ask"]
    )

    out["parity_forward_upper_proxy"] = (
        out["strike"] + out["call_ask"] - out["put_bid"]
    )

    out["parity_forward_interval_width"] = (
        out["parity_forward_upper_proxy"] - out["parity_forward_lower_proxy"]
    )

    out["parity_forward_interval_mid"] = (
        out["parity_forward_lower_proxy"] + out["parity_forward_upper_proxy"]
    ) / 2.0

    out["parity_forward_interval_valid"] = (
        out["parity_forward_lower_proxy"].notna()
        & out["parity_forward_upper_proxy"].notna()
        & (out["parity_forward_lower_proxy"] <= out["parity_forward_upper_proxy"])
    )

    out["parity_forward_mid_positive"] = out["parity_forward_mid_proxy"] > 0

    out["parity_forward_mid_to_spot"] = (
        out["parity_forward_mid_proxy"] / out["underlying_spot_snapshot"]
    )

    out["parity_forward_log_moneyness_proxy"] = np.where(
        (out["strike"] > 0) & (out["parity_forward_mid_proxy"] > 0),
        np.log(out["strike"] / out["parity_forward_mid_proxy"]),
        np.nan,
    )

    out["parity_carry_rate_proxy"] = np.where(
        (out["parity_forward_mid_proxy"] > 0)
        & (out["underlying_spot_snapshot"] > 0)
        & (out["tau_years"] > 0),
        np.log(out["parity_forward_mid_proxy"] / out["underlying_spot_snapshot"])
        / out["tau_years"],
        np.nan,
    )

    # --------------------------------------------------------
    # Expiry-level robust forward estimate
    # --------------------------------------------------------

    expiry_forward_summary = (
        out.groupby(["expiry", "dte_calendar", "tau_years"], dropna=False)
        .agg(
            matched_pairs=("pair_id", "count"),
            median_parity_forward_proxy=("parity_forward_mid_proxy", "median"),
            mean_parity_forward_proxy=("parity_forward_mid_proxy", "mean"),
            min_parity_forward_proxy=("parity_forward_mid_proxy", "min"),
            max_parity_forward_proxy=("parity_forward_mid_proxy", "max"),
            mad_parity_forward_proxy=("parity_forward_mid_proxy", median_absolute_deviation),
            median_parity_forward_interval_width=("parity_forward_interval_width", "median"),
            median_pair_spread_sum=("pair_spread_sum", "median"),
            median_pair_max_rel_spread_mid=("pair_max_rel_spread_mid", "median"),
            median_carry_rate_proxy=("parity_carry_rate_proxy", "median"),
            underlying_spot_snapshot=("underlying_spot_snapshot", "median"),
            positive_forward_pairs=("parity_forward_mid_positive", "sum"),
            valid_forward_interval_pairs=("parity_forward_interval_valid", "sum"),
        )
        .reset_index()
        .sort_values(["dte_calendar", "expiry"])
        .reset_index(drop=True)
    )

    expiry_forward_summary["forward_vs_spot_ratio"] = (
        expiry_forward_summary["median_parity_forward_proxy"]
        / expiry_forward_summary["underlying_spot_snapshot"]
    )

    expiry_forward_summary["forward_minus_spot"] = (
        expiry_forward_summary["median_parity_forward_proxy"]
        - expiry_forward_summary["underlying_spot_snapshot"]
    )

    expiry_forward_summary["forward_range"] = (
        expiry_forward_summary["max_parity_forward_proxy"]
        - expiry_forward_summary["min_parity_forward_proxy"]
    )

    expiry_forward_summary["forward_iqr_proxy"] = (
        out.groupby(["expiry", "dte_calendar", "tau_years"], dropna=False)["parity_forward_mid_proxy"]
        .quantile(0.75)
        .reset_index(name="q75")
        .merge(
            out.groupby(["expiry", "dte_calendar", "tau_years"], dropna=False)["parity_forward_mid_proxy"]
            .quantile(0.25)
            .reset_index(name="q25"),
            on=["expiry", "dte_calendar", "tau_years"],
            how="left",
        )
        .assign(forward_iqr_proxy=lambda x: x["q75"] - x["q25"])
        ["forward_iqr_proxy"]
        .values
    )

    expiry_forward_summary["positive_forward_rate"] = (
        expiry_forward_summary["positive_forward_pairs"]
        / expiry_forward_summary["matched_pairs"]
    )

    expiry_forward_summary["valid_forward_interval_rate"] = (
        expiry_forward_summary["valid_forward_interval_pairs"]
        / expiry_forward_summary["matched_pairs"]
    )

    expiry_forward_summary["usable_forward_proxy"] = (
        (expiry_forward_summary["matched_pairs"] >= config.min_matched_pairs_per_expiry)
        & (expiry_forward_summary["positive_forward_rate"] >= 0.95)
        & (expiry_forward_summary["valid_forward_interval_rate"] >= 0.95)
        & expiry_forward_summary["median_parity_forward_proxy"].notna()
        & (expiry_forward_summary["median_parity_forward_proxy"] > 0)
    )

    # --------------------------------------------------------
    # Merge expiry-level forward estimate back to pair rows
    # --------------------------------------------------------

    merge_columns = [
        "expiry",
        "dte_calendar",
        "tau_years",
        "median_parity_forward_proxy",
        "mad_parity_forward_proxy",
        "forward_iqr_proxy",
        "median_parity_forward_interval_width",
        "median_carry_rate_proxy",
        "forward_vs_spot_ratio",
        "usable_forward_proxy",
    ]

    out = out.merge(
        expiry_forward_summary[merge_columns],
        on=["expiry", "dte_calendar", "tau_years"],
        how="left",
        validate="many_to_one",
    )

    out["parity_forward_residual"] = (
        out["parity_forward_mid_proxy"] - out["median_parity_forward_proxy"]
    )

    out["abs_parity_forward_residual"] = out["parity_forward_residual"].abs()

    out["parity_forward_residual_bps_spot"] = (
        10_000.0
        * out["parity_forward_residual"]
        / out["underlying_spot_snapshot"]
    )

    out["abs_parity_forward_residual_bps_spot"] = (
        out["parity_forward_residual_bps_spot"].abs()
    )

    # Bid-ask-aware tolerance: at least the configured soft tolerance,
    # but wider when the pair's own quote interval is wide.
    out["pair_parity_soft_tolerance"] = np.maximum(
        config.parity_soft_tol_abs,
        out["parity_forward_interval_width"].fillna(0.0) / 2.0,
    )

    # Robust expiry-level tolerance:
    # Use 5 * MAD when available; otherwise fall back to the configured absolute tolerance.
    out["expiry_parity_robust_tolerance"] = np.maximum(
        config.parity_soft_tol_abs,
        5.0 * out["mad_parity_forward_proxy"].fillna(0.0),
    )

    out["parity_forward_outlier_pair_tolerance"] = (
        out["abs_parity_forward_residual"] > out["pair_parity_soft_tolerance"]
    )

    out["parity_forward_outlier_expiry_tolerance"] = (
        out["abs_parity_forward_residual"] > out["expiry_parity_robust_tolerance"]
    )

    out["parity_forward_outlier"] = (
        out["parity_forward_outlier_pair_tolerance"]
        & out["parity_forward_outlier_expiry_tolerance"]
    )

    out["forward_log_moneyness"] = np.where(
        (out["strike"] > 0) & (out["median_parity_forward_proxy"] > 0),
        np.log(out["strike"] / out["median_parity_forward_proxy"]),
        np.nan,
    )

    out["abs_forward_log_moneyness"] = out["forward_log_moneyness"].abs()

    return out, expiry_forward_summary


def summarize_parity_outliers(parity_pairs: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize parity-forward outliers by expiry.
    """
    summary = (
        parity_pairs.groupby(["expiry", "dte_calendar"], dropna=False)
        .agg(
            matched_pairs=("pair_id", "count"),
            parity_forward_outliers=("parity_forward_outlier", "sum"),
            median_abs_forward_residual=("abs_parity_forward_residual", "median"),
            max_abs_forward_residual=("abs_parity_forward_residual", "max"),
            median_abs_forward_residual_bps_spot=("abs_parity_forward_residual_bps_spot", "median"),
            max_abs_forward_residual_bps_spot=("abs_parity_forward_residual_bps_spot", "max"),
            median_pair_parity_soft_tolerance=("pair_parity_soft_tolerance", "median"),
            median_expiry_parity_robust_tolerance=("expiry_parity_robust_tolerance", "median"),
            usable_forward_proxy=("usable_forward_proxy", "first"),
        )
        .reset_index()
        .sort_values(["dte_calendar", "expiry"])
        .reset_index(drop=True)
    )

    summary["parity_forward_outlier_rate"] = (
        summary["parity_forward_outliers"] / summary["matched_pairs"]
    )

    return summary


matched_pairs_with_parity, expiry_forward_summary = add_parity_forward_diagnostics(
    matched_df=matched_call_put_pairs,
    config=CONFIG,
)

parity_outlier_summary = summarize_parity_outliers(matched_pairs_with_parity)

# Update the matched-pairs artifact with parity diagnostics included.
actual_matched_pairs_path = safe_write_dataframe(
    matched_pairs_with_parity,
    ARTIFACT_PATHS["matched_pairs"],
)

ARTIFACT_PATHS["matched_pairs"] = actual_matched_pairs_path
MANIFEST["artifact_paths"]["matched_pairs"] = str(actual_matched_pairs_path)

MANIFEST["parity_diagnostics_completed"] = True
MANIFEST["usable_forward_expiry_count"] = int(expiry_forward_summary["usable_forward_proxy"].sum())
MANIFEST["total_parity_forward_outliers"] = int(matched_pairs_with_parity["parity_forward_outlier"].sum())
MANIFEST["parity_forward_outlier_rate"] = float(
    matched_pairs_with_parity["parity_forward_outlier"].mean()
)
MANIFEST["median_parity_forward_proxy_by_expiry"] = {
    str(row["expiry"]): float(row["median_parity_forward_proxy"])
    for _, row in expiry_forward_summary.iterrows()
}

print("Put-call parity-implied forward diagnostics completed.")
print(f"Matched pairs with parity diagnostics: {len(matched_pairs_with_parity):,}")
print(f"Usable forward expiries: {MANIFEST['usable_forward_expiry_count']:,} / {expiry_forward_summary.shape[0]:,}")
print(f"Parity-forward outliers: {MANIFEST['total_parity_forward_outliers']:,}")
print(f"Parity-forward outlier rate: {MANIFEST['parity_forward_outlier_rate']:.2%}")
print(f"Updated matched pairs path: {actual_matched_pairs_path}")

print("Expiry-level parity-implied forward summary:")
display(expiry_forward_summary)

print("Parity-forward outlier summary:")
display(parity_outlier_summary)

print("Matched pairs with parity diagnostics preview:")
display(
    matched_pairs_with_parity[
        [
            "pair_id",
            "expiry",
            "dte_calendar",
            "strike",
            "underlying_spot_snapshot",
            "parity_forward_mid_proxy",
            "median_parity_forward_proxy",
            "forward_vs_spot_ratio",
            "parity_forward_residual",
            "abs_parity_forward_residual_bps_spot",
            "parity_forward_interval_width",
            "pair_parity_soft_tolerance",
            "expiry_parity_robust_tolerance",
            "parity_forward_outlier",
            "forward_log_moneyness",
            "abs_forward_log_moneyness",
        ]
    ].head(20)
)

Put-call parity-implied forward diagnostics completed.
Matched pairs with parity diagnostics: 1,510
Usable forward expiries: 12 / 12
Parity-forward outliers: 96
Parity-forward outlier rate: 6.36%
Updated matched pairs path: d:\Derivative Pricing Project v1.0+\V1.1\data\processed\spy_option_matched_pairs_20260705_154512_UTC.parquet
Expiry-level parity-implied forward summary:


,expiry,dte_calendar,tau_years,matched_pairs,median_parity_forward_proxy,mean_parity_forward_proxy,min_parity_forward_proxy,max_parity_forward_proxy,mad_parity_forward_proxy,median_parity_forward_interval_width,median_pair_spread_sum,median_pair_max_rel_spread_mid,median_carry_rate_proxy,underlying_spot_snapshot,positive_forward_pairs,valid_forward_interval_pairs,forward_vs_spot_ratio,forward_minus_spot,forward_range,forward_iqr_proxy,positive_forward_rate,valid_forward_interval_rate,usable_forward_proxy
0,2026-07-06,1,0.002740,42,744.557500,744.531667,744.290000,744.745000,0.027500,0.845000,0.845000,0.093495,-0.109073,744.780029,42,42,0.999701,-0.222529,0.455000,0.035000,1.000000,1.000000,True
1,2026-07-07,2,0.005479,46,744.565000,744.563043,744.360000,744.670000,0.075000,0.730000,0.730000,0.061484,-0.052698,744.780029,46,46,0.999711,-0.215029,0.310000,0.143750,1.000000,1.000000,True
2,2026-07-08,3,0.008219,45,744.695000,744.678667,744.390000,744.760000,0.050000,0.680000,0.680000,0.049261,-0.013891,744.780029,45,45,0.999886,-0.085029,0.370000,0.095000,1.000000,1.000000,True
3,2026-07-09,4,0.010959,37,744.800000,744.770135,744.425000,744.840000,0.040000,0.610000,0.610000,0.039260,0.002447,744.780029,37,37,1.000027,0.019971,0.415000,0.075000,1.000000,1.000000,True
4,2026-07-10,5,0.013699,130,745.030000,744.953692,744.390000,745.610000,0.050000,3.070000,3.070000,0.103899,0.024497,744.780029,130,130,1.000336,0.249971,1.220000,0.158750,1.000000,1.000000,True
5,2026-07-17,12,0.032877,178,745.515000,745.351798,744.525000,746.065000,0.115000,3.520000,3.520000,0.102056,0.030001,744.780029,178,178,1.000987,0.734971,1.540000,0.307500,1.000000,1.000000,True
6,2026-07-24,19,0.052055,130,746.072500,745.913538,744.605000,746.460000,0.110000,3.520000,3.520000,0.095885,0.033309,744.780029,130,130,1.001735,1.292471,1.855000,0.305000,1.000000,1.000000,True
7,2026-07-31,26,0.071233,216,746.590000,746.156273,715.120000,755.810000,0.210000,3.285000,3.285000,0.070123,0.034075,744.780029,216,216,1.002430,1.809971,40.690000,0.662500,1.000000,1.000000,True
8,2026-08-07,33,0.090411,50,747.330000,747.162700,744.745000,747.575000,0.060000,3.405000,3.405000,0.105877,0.037805,744.780029,50,50,1.003424,2.549971,2.830000,0.141250,1.000000,1.000000,True
9,2026-08-21,47,0.128767,202,748.247500,747.778045,740.335000,755.575000,0.222500,3.520000,3.520000,0.058807,0.036072,744.780029,202,202,1.004656,3.467471,15.240000,0.953750,1.000000,1.000000,True


Parity-forward outlier summary:


,expiry,dte_calendar,matched_pairs,parity_forward_outliers,median_abs_forward_residual,max_abs_forward_residual,median_abs_forward_residual_bps_spot,max_abs_forward_residual_bps_spot,median_pair_parity_soft_tolerance,median_expiry_parity_robust_tolerance,usable_forward_proxy,parity_forward_outlier_rate
0,2026-07-06,1,42,0,0.027500,0.267500,0.369237,3.591665,0.422500,0.250000,True,0.000000
1,2026-07-07,2,46,0,0.075000,0.205000,1.007009,2.752491,0.365000,0.375000,True,0.000000
2,2026-07-08,3,45,0,0.050000,0.305000,0.671339,4.095169,0.340000,0.250000,True,0.000000
3,2026-07-09,4,37,0,0.040000,0.375000,0.537071,5.035044,0.305000,0.250000,True,0.000000
4,2026-07-10,5,130,0,0.050000,0.640000,0.671339,8.593141,1.535000,0.250000,True,0.000000
5,2026-07-17,12,178,0,0.115000,0.990000,1.544080,13.292515,1.760000,0.575000,True,0.000000
6,2026-07-24,19,130,1,0.110000,1.467500,1.476946,19.703804,1.760000,0.550000,True,0.007692
7,2026-07-31,26,216,18,0.210000,31.470000,2.819624,422.540868,1.642500,1.050000,True,0.083333
8,2026-08-07,33,50,2,0.060000,2.585000,0.805607,34.708235,1.702500,0.300000,True,0.040000
9,2026-08-21,47,202,19,0.222500,7.912500,2.987459,106.239422,1.760000,1.112500,True,0.094059


Matched pairs with parity diagnostics preview:


,pair_id,expiry,dte_calendar,strike,underlying_spot_snapshot,parity_forward_mid_proxy,median_parity_forward_proxy,forward_vs_spot_ratio,parity_forward_residual,abs_parity_forward_residual_bps_spot,parity_forward_interval_width,pair_parity_soft_tolerance,expiry_parity_robust_tolerance,parity_forward_outlier,forward_log_moneyness,abs_forward_log_moneyness
0,0,2026-07-06,1,712.000000,744.780029,744.585000,744.557500,0.999701,0.027500,0.369237,3.530000,1.765000,0.250000,False,-0.044712,0.044712
1,1,2026-07-06,1,713.000000,744.780029,744.585000,744.557500,0.999701,0.027500,0.369237,3.530000,1.765000,0.250000,False,-0.043309,0.043309
2,2,2026-07-06,1,714.000000,744.780029,744.585000,744.557500,0.999701,0.027500,0.369237,3.530000,1.765000,0.250000,False,-0.041907,0.041907
3,3,2026-07-06,1,715.000000,744.780029,744.585000,744.557500,0.999701,0.027500,0.369237,3.530000,1.765000,0.250000,False,-0.040508,0.040508
4,4,2026-07-06,1,717.000000,744.780029,744.590000,744.557500,0.999701,0.032500,0.436370,3.520000,1.760000,0.250000,False,-0.037714,0.037714
5,5,2026-07-06,1,720.000000,744.780029,744.745000,744.557500,0.999701,0.187500,2.517522,2.010000,1.005000,0.250000,False,-0.033539,0.033539
6,6,2026-07-06,1,721.000000,744.780029,744.590000,744.557500,0.999701,0.032500,0.436370,3.520000,1.760000,0.250000,False,-0.032151,0.032151
7,7,2026-07-06,1,722.000000,744.780029,744.595000,744.557500,0.999701,0.037500,0.503504,3.530000,1.765000,0.250000,False,-0.030765,0.030765
8,8,2026-07-06,1,723.000000,744.780029,744.340000,744.557500,0.999701,-0.217500,2.920325,3.040000,1.520000,0.250000,False,-0.029381,0.029381
9,9,2026-07-06,1,724.000000,744.780029,744.390000,744.557500,0.999701,-0.167500,2.248986,1.780000,0.890000,0.250000,False,-0.027999,0.027999


In [8]:
# ============================================================
# Cell 9: Expiry eligibility screen and Notebook 09 candidate panel
# ============================================================

def coerce_expiry_date_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize expiry to Python date objects so all expiry-level tables merge cleanly.
    """
    out = df.copy()

    if "expiry" in out.columns:
        out["expiry"] = pd.to_datetime(out["expiry"], errors="coerce").dt.date

    return out


def safe_bool_series(series: pd.Series, default: bool = False) -> pd.Series:
    """
    Convert a possibly missing/object boolean-like series into a clean boolean series.
    """
    if series is None:
        return pd.Series(dtype=bool)

    return series.fillna(default).astype(bool)


def build_expiry_eligibility_summary(
    quote_filtered_df: pd.DataFrame,
    matched_parity_df: pd.DataFrame,
    expiry_rejection_df: pd.DataFrame,
    matching_df: pd.DataFrame,
    forward_df: pd.DataFrame,
    parity_outlier_df: pd.DataFrame,
    config: Notebook08Config,
) -> pd.DataFrame:
    """
    Build the expiry-level eligibility table for Notebook 09.

    Design rule:
        This function does not rely on ambiguous columns created by repeated merges.
        Every incoming summary is reduced to explicitly named columns before merging.
    """
    filtered = coerce_expiry_date_column(quote_filtered_df)
    matched = coerce_expiry_date_column(matched_parity_df)
    rejection = coerce_expiry_date_column(expiry_rejection_df)
    matching = coerce_expiry_date_column(matching_df)
    forward = coerce_expiry_date_column(forward_df)
    parity = coerce_expiry_date_column(parity_outlier_df)

    # --------------------------------------------------------
    # 1. Clean quote counts by expiry and option type
    # --------------------------------------------------------

    clean_counts = (
        filtered.groupby(["expiry", "dte_calendar", "option_type"], dropna=False)
        .size()
        .rename("clean_rows")
        .reset_index()
        .pivot_table(
            index=["expiry", "dte_calendar"],
            columns="option_type",
            values="clean_rows",
            fill_value=0,
            aggfunc="sum",
        )
        .reset_index()
        .rename(columns={"call": "clean_call_rows", "put": "clean_put_rows"})
    )

    for col in ["clean_call_rows", "clean_put_rows"]:
        if col not in clean_counts.columns:
            clean_counts[col] = 0

    clean_counts["clean_call_rows"] = clean_counts["clean_call_rows"].astype(int)
    clean_counts["clean_put_rows"] = clean_counts["clean_put_rows"].astype(int)
    clean_counts["clean_total_rows"] = (
        clean_counts["clean_call_rows"] + clean_counts["clean_put_rows"]
    )

    # --------------------------------------------------------
    # 2. Strike and quote coverage from the filtered panel
    # --------------------------------------------------------

    filtered_coverage = (
        filtered.groupby(["expiry", "dte_calendar"], dropna=False)
        .agg(
            clean_strike_count=("strike", "nunique"),
            clean_strike_min=("strike", "min"),
            clean_strike_max=("strike", "max"),
            clean_moneyness_min=("moneyness_spot", "min"),
            clean_moneyness_max=("moneyness_spot", "max"),
            median_clean_mid=("mid", "median"),
            median_clean_spread=("spread", "median"),
            median_clean_rel_spread_mid=("rel_spread_mid", "median"),
            max_clean_rel_spread_mid=("rel_spread_mid", "max"),
            warning_rows=("warning_flag_count", lambda x: int((x > 0).sum())),
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # 3. Matched-pair coverage computed directly from matched pairs
    # --------------------------------------------------------

    matched_coverage = (
        matched.groupby(["expiry", "dte_calendar"], dropna=False)
        .agg(
            matched_pair_count=("pair_id", "count"),
            matched_strike_count=("strike", "nunique"),
            matched_strike_min=("strike", "min"),
            matched_strike_max=("strike", "max"),
            matched_moneyness_min=("moneyness_spot", "min"),
            matched_moneyness_max=("moneyness_spot", "max"),
            median_pair_max_rel_spread_mid=("pair_max_rel_spread_mid", "median"),
            median_pair_mean_rel_spread_mid=("pair_mean_rel_spread_mid", "median"),
            pair_warning_rows=("pair_warning_flag_count", lambda x: int((x > 0).sum())),
            pairs_with_both_volume=("both_sides_have_volume", "sum"),
            pairs_with_both_open_interest=("both_sides_have_open_interest", "sum"),
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # 4. Matching summary, explicitly renamed
    # --------------------------------------------------------

    matching_keep = [
        "expiry",
        "dte_calendar",
        "call_match_rate",
        "put_match_rate",
        "min_side_match_rate",
    ]

    matching_renamed = matching[
        [col for col in matching_keep if col in matching.columns]
    ].copy()

    # --------------------------------------------------------
    # 5. Rejection summary, explicitly renamed
    # --------------------------------------------------------

    rejection_keep = [
        "expiry",
        "dte_calendar",
        "raw_rows",
        "passed_rows",
        "rejected_rows",
        "pass_rate",
        "wide_rel_spread_rows",
        "dust_mid_rows",
        "crossed_market_rows",
        "soft_warning_rows",
    ]

    rejection_renamed = rejection[
        [col for col in rejection_keep if col in rejection.columns]
    ].copy()

    rejection_renamed = rejection_renamed.rename(
        columns={
            "raw_rows": "raw_quote_rows",
            "passed_rows": "quote_quality_passed_rows",
            "rejected_rows": "quote_quality_rejected_rows",
            "pass_rate": "quote_quality_pass_rate",
        }
    )

    # --------------------------------------------------------
    # 6. Forward summary, explicitly renamed
    # --------------------------------------------------------

    forward_keep = [
        "expiry",
        "dte_calendar",
        "tau_years",
        "matched_pairs",
        "median_parity_forward_proxy",
        "mean_parity_forward_proxy",
        "mad_parity_forward_proxy",
        "forward_iqr_proxy",
        "forward_range",
        "median_parity_forward_interval_width",
        "median_carry_rate_proxy",
        "underlying_spot_snapshot",
        "forward_vs_spot_ratio",
        "forward_minus_spot",
        "positive_forward_rate",
        "valid_forward_interval_rate",
        "usable_forward_proxy",
    ]

    forward_renamed = forward[
        [col for col in forward_keep if col in forward.columns]
    ].copy()

    forward_renamed = forward_renamed.rename(
        columns={
            "matched_pairs": "forward_matched_pair_count",
        }
    )

    # --------------------------------------------------------
    # 7. Parity outlier summary, explicitly renamed
    # --------------------------------------------------------

    parity_keep = [
        "expiry",
        "dte_calendar",
        "matched_pairs",
        "parity_forward_outliers",
        "parity_forward_outlier_rate",
        "median_abs_forward_residual",
        "max_abs_forward_residual",
        "median_abs_forward_residual_bps_spot",
        "max_abs_forward_residual_bps_spot",
        "usable_forward_proxy",
    ]

    parity_renamed = parity[
        [col for col in parity_keep if col in parity.columns]
    ].copy()

    parity_renamed = parity_renamed.rename(
        columns={
            "matched_pairs": "parity_matched_pair_count",
            "usable_forward_proxy": "parity_usable_forward_proxy",
        }
    )

    # --------------------------------------------------------
    # 8. Merge all expiry-level summaries
    # --------------------------------------------------------

    summary = clean_counts.merge(
        filtered_coverage,
        on=["expiry", "dte_calendar"],
        how="left",
        validate="one_to_one",
    )

    for table in [
        matched_coverage,
        matching_renamed,
        rejection_renamed,
        forward_renamed,
        parity_renamed,
    ]:
        if not table.empty:
            summary = summary.merge(
                table,
                on=["expiry", "dte_calendar"],
                how="left",
                validate="one_to_one",
            )

    # --------------------------------------------------------
    # 9. Fill safe defaults for count columns
    # --------------------------------------------------------

    count_columns = [
        "matched_pair_count",
        "matched_strike_count",
        "pair_warning_rows",
        "pairs_with_both_volume",
        "pairs_with_both_open_interest",
        "raw_quote_rows",
        "quote_quality_passed_rows",
        "quote_quality_rejected_rows",
        "wide_rel_spread_rows",
        "dust_mid_rows",
        "crossed_market_rows",
        "soft_warning_rows",
        "forward_matched_pair_count",
        "parity_matched_pair_count",
        "parity_forward_outliers",
    ]

    for col in count_columns:
        if col not in summary.columns:
            summary[col] = 0
        summary[col] = summary[col].fillna(0).astype(int)

    bool_columns = [
        "usable_forward_proxy",
        "parity_usable_forward_proxy",
    ]

    for col in bool_columns:
        if col not in summary.columns:
            summary[col] = False
        summary[col] = summary[col].fillna(False).astype(bool)

    # --------------------------------------------------------
    # 10. Eligibility tests for Notebook 09
    # --------------------------------------------------------

    summary["passes_min_clean_calls"] = (
        summary["clean_call_rows"] >= config.min_clean_calls_per_expiry
    )

    summary["passes_min_clean_puts"] = (
        summary["clean_put_rows"] >= config.min_clean_puts_per_expiry
    )

    summary["passes_min_matched_pairs"] = (
        summary["matched_pair_count"] >= config.min_matched_pairs_per_expiry
    )

    summary["passes_positive_forward_proxy"] = (
        summary["median_parity_forward_proxy"].notna()
        & (summary["median_parity_forward_proxy"] > 0)
    )

    summary["passes_forward_usability"] = summary["usable_forward_proxy"]

    summary["passes_no_crossed_markets"] = (
        summary["crossed_market_rows"].fillna(0).astype(int) == 0
    )

    summary["passes_strike_coverage"] = (
        summary["clean_strike_count"].fillna(0).astype(int)
        >= config.min_matched_pairs_per_expiry
    )

    summary["passes_quote_quality_nonempty"] = (
        summary["clean_total_rows"].fillna(0).astype(int) > 0
    )

    # Soft condition: parity outliers are allowed, but not if they dominate the expiry.
    if "parity_forward_outlier_rate" not in summary.columns:
        summary["parity_forward_outlier_rate"] = np.nan

    summary["passes_parity_outlier_soft_screen"] = (
        summary["parity_forward_outlier_rate"].fillna(0.0) <= 0.25
    )

    eligibility_columns = [
        "passes_quote_quality_nonempty",
        "passes_min_clean_calls",
        "passes_min_clean_puts",
        "passes_min_matched_pairs",
        "passes_strike_coverage",
        "passes_positive_forward_proxy",
        "passes_forward_usability",
        "passes_no_crossed_markets",
        "passes_parity_outlier_soft_screen",
    ]

    summary["expiry_eligibility_fail_count"] = (
        ~summary[eligibility_columns].fillna(False).astype(bool)
    ).sum(axis=1).astype(int)

    summary["expiry_pass_for_09"] = summary["expiry_eligibility_fail_count"] == 0

    # --------------------------------------------------------
    # 11. Human-readable expiry status
    # --------------------------------------------------------

    status = np.full(len(summary), "PASS_FOR_09", dtype=object)

    status[~summary["passes_min_matched_pairs"]] = "FAIL_INSUFFICIENT_MATCHED_PAIRS"
    status[~summary["passes_min_clean_puts"]] = "FAIL_INSUFFICIENT_CLEAN_PUTS"
    status[~summary["passes_min_clean_calls"]] = "FAIL_INSUFFICIENT_CLEAN_CALLS"
    status[~summary["passes_forward_usability"]] = "FAIL_FORWARD_PROXY_NOT_USABLE"
    status[~summary["passes_parity_outlier_soft_screen"]] = "CONDITIONAL_HIGH_PARITY_OUTLIERS"

    summary["expiry_status_for_09"] = np.where(
        summary["expiry_pass_for_09"],
        "PASS_FOR_09",
        status,
    )

    # Keep a readable column order.
    front_columns = [
        "expiry",
        "dte_calendar",
        "tau_years",
        "expiry_pass_for_09",
        "expiry_status_for_09",
        "expiry_eligibility_fail_count",
        "clean_call_rows",
        "clean_put_rows",
        "clean_total_rows",
        "matched_pair_count",
        "clean_strike_count",
        "matched_strike_count",
        "clean_strike_min",
        "clean_strike_max",
        "matched_strike_min",
        "matched_strike_max",
        "median_parity_forward_proxy",
        "forward_vs_spot_ratio",
        "forward_minus_spot",
        "usable_forward_proxy",
        "parity_forward_outliers",
        "parity_forward_outlier_rate",
        "quote_quality_pass_rate",
    ]

    front_columns = [col for col in front_columns if col in summary.columns]
    remaining_columns = [col for col in summary.columns if col not in front_columns]

    summary = summary[front_columns + remaining_columns].sort_values(
        ["dte_calendar", "expiry"]
    ).reset_index(drop=True)

    return summary


def build_notebook09_candidate_panel(
    quote_filtered_df: pd.DataFrame,
    expiry_summary_df: pd.DataFrame,
    forward_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build the filtered option quote panel to hand off to Notebook 09.

    One row = one clean option quote from an expiry that passed the Notebook 09
    eligibility screen.
    """
    filtered = coerce_expiry_date_column(quote_filtered_df)
    expiry_summary = coerce_expiry_date_column(expiry_summary_df)
    forward = coerce_expiry_date_column(forward_df)

    eligible_expiries = expiry_summary.loc[
        expiry_summary["expiry_pass_for_09"], ["expiry", "dte_calendar"]
    ].copy()

    candidate = filtered.merge(
        eligible_expiries,
        on=["expiry", "dte_calendar"],
        how="inner",
        validate="many_to_one",
    )

    forward_columns = [
        "expiry",
        "dte_calendar",
        "median_parity_forward_proxy",
        "forward_vs_spot_ratio",
        "forward_minus_spot",
        "median_carry_rate_proxy",
        "usable_forward_proxy",
    ]

    forward_attach = forward[
        [col for col in forward_columns if col in forward.columns]
    ].copy()

    candidate = candidate.merge(
        forward_attach,
        on=["expiry", "dte_calendar"],
        how="left",
        validate="many_to_one",
    )

    candidate["forward_log_moneyness_proxy"] = np.where(
        (candidate["strike"] > 0)
        & (candidate["median_parity_forward_proxy"] > 0),
        np.log(candidate["strike"] / candidate["median_parity_forward_proxy"]),
        np.nan,
    )

    candidate["abs_forward_log_moneyness_proxy"] = (
        candidate["forward_log_moneyness_proxy"].abs()
    )

    candidate["notebook09_candidate"] = True

    candidate = candidate.sort_values(
        ["expiry", "option_type", "strike"]
    ).reset_index(drop=True)

    return candidate


expiry_eligibility_summary = build_expiry_eligibility_summary(
    quote_filtered_df=quote_filtered_panel,
    matched_parity_df=matched_pairs_with_parity,
    expiry_rejection_df=expiry_rejection_summary,
    matching_df=matching_expiry_summary,
    forward_df=expiry_forward_summary,
    parity_outlier_df=parity_outlier_summary,
    config=CONFIG,
)

notebook09_candidate_panel = build_notebook09_candidate_panel(
    quote_filtered_df=quote_filtered_panel,
    expiry_summary_df=expiry_eligibility_summary,
    forward_df=expiry_forward_summary,
)

# ------------------------------------------------------------
# Save expiry summary and Notebook 09 candidate panel
# ------------------------------------------------------------

actual_candidate_panel_path = safe_write_dataframe(
    notebook09_candidate_panel,
    ARTIFACT_PATHS["n09_candidate_panel"],
)

ARTIFACT_PATHS["n09_candidate_panel"] = actual_candidate_panel_path
MANIFEST["artifact_paths"]["n09_candidate_panel"] = str(actual_candidate_panel_path)

actual_expiry_summary_path = ARTIFACT_PATHS["expiry_summary"]
actual_expiry_summary_path.parent.mkdir(parents=True, exist_ok=True)
expiry_eligibility_summary.to_csv(actual_expiry_summary_path, index=False)

MANIFEST["artifact_paths"]["expiry_summary"] = str(actual_expiry_summary_path)

MANIFEST["expiry_eligibility_completed"] = True
MANIFEST["expiry_count_total"] = int(len(expiry_eligibility_summary))
MANIFEST["expiry_count_pass_for_09"] = int(expiry_eligibility_summary["expiry_pass_for_09"].sum())
MANIFEST["notebook09_candidate_rows"] = int(len(notebook09_candidate_panel))
MANIFEST["notebook09_candidate_call_rows"] = int(
    (notebook09_candidate_panel["option_type"] == "call").sum()
)
MANIFEST["notebook09_candidate_put_rows"] = int(
    (notebook09_candidate_panel["option_type"] == "put").sum()
)
MANIFEST["notebook09_candidate_expiries"] = [
    str(x)
    for x in sorted(notebook09_candidate_panel["expiry"].dropna().unique())
]

print("Expiry eligibility screen completed.")
print(f"Total expiries screened: {MANIFEST['expiry_count_total']:,}")
print(f"Expiries passing for Notebook 09: {MANIFEST['expiry_count_pass_for_09']:,}")
print(f"Notebook 09 candidate rows: {MANIFEST['notebook09_candidate_rows']:,}")
print(f"Candidate call rows: {MANIFEST['notebook09_candidate_call_rows']:,}")
print(f"Candidate put rows: {MANIFEST['notebook09_candidate_put_rows']:,}")
print(f"Candidate panel path: {actual_candidate_panel_path}")
print(f"Expiry summary path: {actual_expiry_summary_path}")

print("Expiry eligibility summary:")
display(expiry_eligibility_summary)

print("Notebook 09 candidate panel preview:")
display(
    notebook09_candidate_panel[
        [
            "row_id",
            "option_type",
            "expiry",
            "dte_calendar",
            "tau_years",
            "strike",
            "moneyness_spot",
            "forward_log_moneyness_proxy",
            "bid",
            "ask",
            "mid",
            "spread",
            "rel_spread_mid",
            "median_parity_forward_proxy",
            "forward_vs_spot_ratio",
            "warning_flag_count",
            "quote_quality_bucket",
            "notebook09_candidate",
        ]
    ].head(25)
)

Expiry eligibility screen completed.
Total expiries screened: 12
Expiries passing for Notebook 09: 12
Notebook 09 candidate rows: 3,527
Candidate call rows: 1,744
Candidate put rows: 1,783
Candidate panel path: d:\Derivative Pricing Project v1.0+\V1.1\data\processed\spy_option_n09_candidate_20260705_154512_UTC.parquet
Expiry summary path: d:\Derivative Pricing Project v1.0+\V1.1\data\processed\spy_option_expiry_summary_20260705_154512_UTC.csv
Expiry eligibility summary:


,expiry,dte_calendar,tau_years,expiry_pass_for_09,expiry_status_for_09,expiry_eligibility_fail_count,clean_call_rows,clean_put_rows,clean_total_rows,matched_pair_count,clean_strike_count,matched_strike_count,clean_strike_min,clean_strike_max,matched_strike_min,matched_strike_max,median_parity_forward_proxy,forward_vs_spot_ratio,forward_minus_spot,usable_forward_proxy,parity_forward_outliers,parity_forward_outlier_rate,quote_quality_pass_rate,clean_moneyness_min,clean_moneyness_max,median_clean_mid,median_clean_spread,median_clean_rel_spread_mid,max_clean_rel_spread_mid,warning_rows,matched_moneyness_min,matched_moneyness_max,median_pair_max_rel_spread_mid,median_pair_mean_rel_spread_mid,pair_warning_rows,pairs_with_both_volume,pairs_with_both_open_interest,call_match_rate,put_match_rate,min_side_match_rate,raw_quote_rows,quote_quality_passed_rows,quote_quality_rejected_rows,wide_rel_spread_rows,dust_mid_rows,crossed_market_rows,soft_warning_rows,forward_matched_pair_count,mean_parity_forward_proxy,mad_parity_forward_proxy,forward_iqr_proxy,forward_range,median_parity_forward_interval_width,median_carry_rate_proxy,underlying_spot_snapshot,positive_forward_rate,valid_forward_interval_rate,parity_matched_pair_count,median_abs_forward_residual,max_abs_forward_residual,median_abs_forward_residual_bps_spot,max_abs_forward_residual_bps_spot,parity_usable_forward_proxy,passes_min_clean_calls,passes_min_clean_puts,passes_min_matched_pairs,passes_positive_forward_proxy,passes_forward_usability,passes_no_crossed_markets,passes_strike_coverage,passes_quote_quality_nonempty,passes_parity_outlier_soft_screen
0,2026-07-06,1,0.002740,True,PASS_FOR_09,0,54,52,106,42,64,42,625.000000,777.000000,712.000000,757.000000,744.557500,0.999701,-0.222529,True,0,0.000000,0.527363,0.839174,1.043261,4.435000,0.090000,0.085616,0.400000,8,0.955987,1.016407,0.093495,0.078362,7,41,41,0.777778,0.807692,0.777778,201,106,95,95,73,0,81,42,744.531667,0.027500,0.035000,0.455000,0.845000,-0.109073,744.780029,1.000000,1.000000,42,0.027500,0.267500,0.369237,3.591665,True,True,True,True,True,True,True,True,True,True
1,2026-07-07,2,0.005479,True,PASS_FOR_09,0,50,64,114,46,68,46,680.000000,765.000000,690.000000,762.000000,744.565000,0.999711,-0.215029,True,0,0.000000,0.655172,0.913021,1.027149,1.370000,0.020000,0.058864,0.400000,3,0.926448,1.023121,0.061484,0.048777,2,46,44,0.920000,0.718750,0.718750,174,114,60,60,45,0,48,46,744.563043,0.075000,0.143750,0.310000,0.730000,-0.052698,744.780029,1.000000,1.000000,46,0.075000,0.205000,1.007009,2.752491,True,True,True,True,True,True,True,True,True,True
2,2026-07-08,3,0.008219,True,PASS_FOR_09,0,56,63,119,45,74,45,640.000000,766.000000,675.000000,760.000000,744.695000,0.999886,-0.085029,True,0,0.000000,0.700000,0.859314,1.028492,1.285000,0.020000,0.050312,0.400000,9,0.906308,1.020436,0.049261,0.036245,8,41,44,0.803571,0.714286,0.714286,170,119,51,51,27,0,37,45,744.678667,0.050000,0.095000,0.370000,0.680000,-0.013891,744.780029,1.000000,1.000000,45,0.050000,0.305000,0.671339,4.095169,True,True,True,True,True,True,True,True,True,True
3,2026-07-09,4,0.010959,True,PASS_FOR_09,0,54,63,117,37,80,37,650.000000,770.000000,685.000000,760.000000,744.800000,1.000027,0.019971,True,0,0.000000,0.801370,0.872741,1.033862,1.005000,0.020000,0.039216,0.400000,4,0.919735,1.020436,0.039260,0.029167,4,34,36,0.685185,0.587302,0.587302,146,117,29,29,12,0,16,37,744.770135,0.040000,0.075000,0.415000,0.610000,0.002447,744.780029,1.000000,1.000000,37,0.040000,0.375000,0.537071,5.035044,True,True,True,True,True,True,True,True,True,True
4,2026-07-10,5,0.013699,True,PASS_FOR_09,0,147,149,296,130,166,130,450.000000,805.000000,615.000000,775.000000,745.030000,1.000336,0.249971,True,0,0.000000,0.791444,0.604205,1.080856,6.685000,0.325000,0.054616,0.400000,49,0.825747,1.040576,0.103899,0.079229,27,123,121,0.884354,0.872483,0.872483,374,296,78,78,38,0,87,130,744.953692,0.050000,0.158750,1.220000,3.070000,0.024497,744.780029,1.000000,1.000000,130,0.050000,0.640000

Notebook 09 candidate panel preview:


,row_id,option_type,expiry,dte_calendar,tau_years,strike,moneyness_spot,forward_log_moneyness_proxy,bid,ask,mid,spread,rel_spread_mid,median_parity_forward_proxy,forward_vs_spot_ratio,warning_flag_count,quote_quality_bucket,notebook09_candidate
0,0,call,2026-07-06,1,0.002740,625.000000,0.839174,-0.175038,117.820000,121.340000,119.580000,3.520000,0.029436,744.557500,0.999701,0,clean_candidate,True
1,1,call,2026-07-06,1,0.002740,660.000000,0.886168,-0.120550,82.830000,86.350000,84.590000,3.520000,0.041612,744.557500,0.999701,0,clean_candidate,True
2,2,call,2026-07-06,1,0.002740,665.000000,0.892881,-0.113003,77.830000,81.350000,79.590000,3.520000,0.044227,744.557500,0.999701,0,clean_candidate,True
3,3,call,2026-07-06,1,0.002740,670.000000,0.899594,-0.105512,72.830000,76.350000,74.590000,3.520000,0.047191,744.557500,0.999701,0,clean_candidate,True
4,4,call,2026-07-06,1,0.002740,700.000000,0.939875,-0.061710,42.840000,46.360000,44.600000,3.520000,0.078924,744.557500,0.999701,0,clean_candidate,True
5,5,call,2026-07-06,1,0.002740,703.000000,0.943903,-0.057433,39.840000,43.360000,41.600000,3.520000,0.084615,744.557500,0.999701,0,clean_candidate,True
6,6,call,2026-07-06,1,0.002740,705.000000,0.946588,-0.054592,37.850000,41.360000,39.605000,3.510000,0.088625,744.557500,0.999701,0,clean_candidate,True
7,7,call,2026-07-06,1,0.002740,706.000000,0.947931,-0.053175,36.890000,40.360000,38.625000,3.470000,0.089838,744.557500,0.999701,0,clean_candidate,True
8,8,call,2026-07-06,1,0.002740,709.000000,0.951959,-0.048935,33.850000,37.360000,35.605000,3.510000,0.098582,744.557500,0.999701,0,clean_candidate,True
9,9,call,2026-07-06,1,0.002740,710.000000,0.953302,-0.047525,32.850000,36.360000,34.605000,3.510000,0.101430,744.557500,0.999701,0,clean_candidate,True


In [9]:
# ============================================================
# Cell 10: Final validation ledger, manifest export, and readiness flag
# ============================================================

def json_safe(value: Any) -> Any:
    """
    Convert common numpy/pandas/path/datetime objects into JSON-safe objects.
    """
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (datetime, pd.Timestamp)):
        return value.isoformat()

    if hasattr(value, "isoformat") and not isinstance(value, str):
        try:
            return value.isoformat()
        except Exception:
            pass

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        if np.isnan(value):
            return None
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}

    if isinstance(value, list):
        return [json_safe(v) for v in value]

    if isinstance(value, tuple):
        return [json_safe(v) for v in value]

    if pd.isna(value) if not isinstance(value, (list, tuple, dict, pd.Series, pd.DataFrame)) else False:
        return None

    return value


def make_validation_record(
    check_name: str,
    passed: bool,
    severity: str,
    observed_value: Any,
    expected_value: Any,
    description: str,
) -> dict[str, Any]:
    """
    Build one final validation-ledger row.
    """
    return {
        "check_name": check_name,
        "status": "PASS" if bool(passed) else "FAIL",
        "severity": severity,
        "observed_value": json_safe(observed_value),
        "expected_value": json_safe(expected_value),
        "description": description,
    }


def build_final_validation_ledger() -> pd.DataFrame:
    """
    Final Notebook 08 validation ledger.

    This ledger checks whether the notebook produced a reproducible,
    quote-quality-controlled SPY option panel that is safe to hand off
    to Notebook 09.

    It does not certify that the market surface is arbitrage-free.
    """
    records = []

    # --------------------------------------------------------
    # Raw acquisition and standardization checks
    # --------------------------------------------------------

    records.append(
        make_validation_record(
            check_name="raw_snapshot_nonempty",
            passed=len(raw_option_snapshot) > 0,
            severity="CRITICAL",
            observed_value=len(raw_option_snapshot),
            expected_value="> 0",
            description="Raw option-chain snapshot must contain rows.",
        )
    )

    records.append(
        make_validation_record(
            check_name="standardized_panel_nonempty",
            passed=len(standardized_raw_panel) > 0,
            severity="CRITICAL",
            observed_value=len(standardized_raw_panel),
            expected_value="> 0",
            description="Standardized raw panel must contain rows.",
        )
    )

    records.append(
        make_validation_record(
            check_name="standardized_row_count_matches_raw",
            passed=len(standardized_raw_panel) == len(raw_option_snapshot),
            severity="CRITICAL",
            observed_value=len(standardized_raw_panel),
            expected_value=len(raw_option_snapshot),
            description="Standardization should not silently drop raw rows.",
        )
    )

    records.append(
        make_validation_record(
            check_name="raw_integrity_checks_pass",
            passed=(raw_integrity_ledger["status"] == "PASS").all(),
            severity="CRITICAL",
            observed_value=raw_integrity_ledger["status"].value_counts().to_dict(),
            expected_value="all PASS",
            description="Raw structural integrity checks must pass.",
        )
    )

    # --------------------------------------------------------
    # Quote-feature and quote-filter checks
    # --------------------------------------------------------

    records.append(
        make_validation_record(
            check_name="quote_feature_panel_row_count_matches_standardized",
            passed=len(quote_feature_panel) == len(standardized_raw_panel),
            severity="CRITICAL",
            observed_value=len(quote_feature_panel),
            expected_value=len(standardized_raw_panel),
            description="Quote-feature engineering should not drop rows.",
        )
    )

    records.append(
        make_validation_record(
            check_name="quote_screened_panel_row_count_matches_standardized",
            passed=len(quote_screened_panel) == len(standardized_raw_panel),
            severity="CRITICAL",
            observed_value=len(quote_screened_panel),
            expected_value=len(standardized_raw_panel),
            description="Quote screening should evaluate every standardized row.",
        )
    )

    records.append(
        make_validation_record(
            check_name="quote_filter_accounting_consistent",
            passed=(len(quote_filtered_panel) + len(quote_rejection_ledger)) == len(quote_screened_panel),
            severity="CRITICAL",
            observed_value={
                "passed": len(quote_filtered_panel),
                "rejected": len(quote_rejection_ledger),
                "screened": len(quote_screened_panel),
            },
            expected_value="passed + rejected == screened",
            description="Filtered rows and rejection ledger must reconcile to screened rows.",
        )
    )

    records.append(
        make_validation_record(
            check_name="quote_filtered_panel_nonempty",
            passed=len(quote_filtered_panel) > 0,
            severity="CRITICAL",
            observed_value=len(quote_filtered_panel),
            expected_value="> 0",
            description="Quote-quality-filtered panel must contain usable option quotes.",
        )
    )

    records.append(
        make_validation_record(
            check_name="filtered_panel_has_calls_and_puts",
            passed=(
                (quote_filtered_panel["option_type"] == "call").sum() > 0
                and (quote_filtered_panel["option_type"] == "put").sum() > 0
            ),
            severity="CRITICAL",
            observed_value=quote_filtered_panel["option_type"].value_counts().to_dict(),
            expected_value="both call and put rows present",
            description="Filtered panel must contain both calls and puts.",
        )
    )

    records.append(
        make_validation_record(
            check_name="no_rejected_rows_in_filtered_panel",
            passed=quote_filtered_panel["is_quote_quality_pass"].all(),
            severity="CRITICAL",
            observed_value=quote_filtered_panel["is_quote_quality_pass"].value_counts().to_dict(),
            expected_value="all True",
            description="Every row in the filtered panel must pass quote-quality rules.",
        )
    )

    # --------------------------------------------------------
    # Matched-pair and parity checks
    # --------------------------------------------------------

    records.append(
        make_validation_record(
            check_name="matched_pairs_nonempty",
            passed=len(matched_pairs_with_parity) > 0,
            severity="CRITICAL",
            observed_value=len(matched_pairs_with_parity),
            expected_value="> 0",
            description="Matched call-put pair panel must contain rows.",
        )
    )

    records.append(
        make_validation_record(
            check_name="matched_pairs_have_positive_forward_proxy",
            passed=(matched_pairs_with_parity["parity_forward_mid_proxy"] > 0).all(),
            severity="CRITICAL",
            observed_value=int((matched_pairs_with_parity["parity_forward_mid_proxy"] <= 0).sum()),
            expected_value=0,
            description="Every matched pair should have a positive parity-forward proxy.",
        )
    )

    records.append(
        make_validation_record(
            check_name="at_least_one_usable_forward_expiry",
            passed=expiry_forward_summary["usable_forward_proxy"].sum() >= 1,
            severity="CRITICAL",
            observed_value=int(expiry_forward_summary["usable_forward_proxy"].sum()),
            expected_value=">= 1",
            description="At least one expiry must have a usable parity-implied forward proxy.",
        )
    )

    records.append(
        make_validation_record(
            check_name="parity_diagnostics_available",
            passed={
                "parity_forward_mid_proxy",
                "median_parity_forward_proxy",
                "parity_forward_outlier",
                "forward_log_moneyness",
            }.issubset(set(matched_pairs_with_parity.columns)),
            severity="CRITICAL",
            observed_value=[
                col for col in [
                    "parity_forward_mid_proxy",
                    "median_parity_forward_proxy",
                    "parity_forward_outlier",
                    "forward_log_moneyness",
                ]
                if col in matched_pairs_with_parity.columns
            ],
            expected_value="required parity diagnostic columns present",
            description="Matched-pair panel must include parity-forward diagnostics.",
        )
    )

    # --------------------------------------------------------
    # Expiry eligibility and Notebook 09 candidate checks
    # --------------------------------------------------------

    records.append(
        make_validation_record(
            check_name="expiry_eligibility_summary_nonempty",
            passed=len(expiry_eligibility_summary) > 0,
            severity="CRITICAL",
            observed_value=len(expiry_eligibility_summary),
            expected_value="> 0",
            description="Expiry-level eligibility summary must contain rows.",
        )
    )

    records.append(
        make_validation_record(
            check_name="at_least_one_expiry_passes_for_09",
            passed=expiry_eligibility_summary["expiry_pass_for_09"].sum() >= 1,
            severity="CRITICAL",
            observed_value=int(expiry_eligibility_summary["expiry_pass_for_09"].sum()),
            expected_value=">= 1",
            description="At least one expiry must pass the Notebook 09 handoff screen.",
        )
    )

    records.append(
        make_validation_record(
            check_name="candidate_panel_nonempty",
            passed=len(notebook09_candidate_panel) > 0,
            severity="CRITICAL",
            observed_value=len(notebook09_candidate_panel),
            expected_value="> 0",
            description="Notebook 09 candidate panel must contain rows.",
        )
    )

    records.append(
        make_validation_record(
            check_name="candidate_panel_has_calls_and_puts",
            passed=(
                (notebook09_candidate_panel["option_type"] == "call").sum() > 0
                and (notebook09_candidate_panel["option_type"] == "put").sum() > 0
            ),
            severity="CRITICAL",
            observed_value=notebook09_candidate_panel["option_type"].value_counts().to_dict(),
            expected_value="both call and put rows present",
            description="Notebook 09 candidate panel must contain both calls and puts.",
        )
    )

    records.append(
        make_validation_record(
            check_name="candidate_panel_has_no_nonpositive_mid",
            passed=(notebook09_candidate_panel["mid"] > 0).all(),
            severity="CRITICAL",
            observed_value=int((notebook09_candidate_panel["mid"] <= 0).sum()),
            expected_value=0,
            description="Candidate panel must not contain nonpositive mid prices.",
        )
    )

    records.append(
        make_validation_record(
            check_name="candidate_panel_has_no_crossed_markets",
            passed=not (notebook09_candidate_panel["ask"] < notebook09_candidate_panel["bid"]).any(),
            severity="CRITICAL",
            observed_value=int((notebook09_candidate_panel["ask"] < notebook09_candidate_panel["bid"]).sum()),
            expected_value=0,
            description="Candidate panel must not contain crossed markets.",
        )
    )

    records.append(
        make_validation_record(
            check_name="candidate_panel_has_forward_log_moneyness",
            passed=notebook09_candidate_panel["forward_log_moneyness_proxy"].notna().any(),
            severity="CRITICAL",
            observed_value=int(notebook09_candidate_panel["forward_log_moneyness_proxy"].notna().sum()),
            expected_value="> 0",
            description="Candidate panel should contain forward-log-moneyness proxy values for Notebook 09.",
        )
    )

    records.append(
        make_validation_record(
            check_name="candidate_expiries_match_passed_expiries",
            passed=(
                set(notebook09_candidate_panel["expiry"].dropna().unique())
                == set(expiry_eligibility_summary.loc[
                    expiry_eligibility_summary["expiry_pass_for_09"], "expiry"
                ].dropna().unique())
            ),
            severity="CRITICAL",
            observed_value=sorted(str(x) for x in notebook09_candidate_panel["expiry"].dropna().unique()),
            expected_value=sorted(
                str(x)
                for x in expiry_eligibility_summary.loc[
                    expiry_eligibility_summary["expiry_pass_for_09"], "expiry"
                ].dropna().unique()
            ),
            description="Candidate panel should contain exactly the expiries that passed the eligibility screen.",
        )
    )

    # --------------------------------------------------------
    # Artifact checks
    # --------------------------------------------------------

    critical_artifact_keys = [
        "raw_snapshot",
        "raw_standardized_panel",
        "quote_filtered_panel",
        "matched_pairs",
        "n09_candidate_panel",
        "rejection_ledger",
        "expiry_summary",
    ]

    for artifact_key in critical_artifact_keys:
        artifact_path = Path(ARTIFACT_PATHS[artifact_key])

        records.append(
            make_validation_record(
                check_name=f"artifact_exists__{artifact_key}",
                passed=artifact_path.exists(),
                severity="CRITICAL",
                observed_value=str(artifact_path),
                expected_value="path exists",
                description=f"Required artifact must exist: {artifact_key}.",
            )
        )

    final_validation_ledger = pd.DataFrame(records)

    severity_rank = {"CRITICAL": 0, "WARNING": 1, "INFO": 2}
    final_validation_ledger["severity_rank"] = (
        final_validation_ledger["severity"].map(severity_rank).fillna(99)
    )

    final_validation_ledger = final_validation_ledger.sort_values(
        ["status", "severity_rank", "check_name"],
        ascending=[True, True, True],
    ).drop(columns=["severity_rank"]).reset_index(drop=True)

    return final_validation_ledger


final_validation_ledger = build_final_validation_ledger()

critical_failures = final_validation_ledger.loc[
    (final_validation_ledger["severity"] == "CRITICAL")
    & (final_validation_ledger["status"] == "FAIL")
].copy()

NOTEBOOK_08_READY_FOR_09 = critical_failures.empty

NOTEBOOK_08_STATUS = (
    "READY_FOR_09"
    if NOTEBOOK_08_READY_FOR_09
    else "NOT_READY_FOR_09"
)

# ------------------------------------------------------------
# Final manifest update
# ------------------------------------------------------------

MANIFEST["final_validation_completed"] = True
MANIFEST["NOTEBOOK_08_READY_FOR_09"] = bool(NOTEBOOK_08_READY_FOR_09)
MANIFEST["NOTEBOOK_08_STATUS"] = NOTEBOOK_08_STATUS
MANIFEST["critical_validation_failures"] = critical_failures["check_name"].tolist()
MANIFEST["final_validation_pass_count"] = int((final_validation_ledger["status"] == "PASS").sum())
MANIFEST["final_validation_fail_count"] = int((final_validation_ledger["status"] == "FAIL").sum())

MANIFEST["final_handoff_summary"] = {
    "raw_rows": int(len(raw_option_snapshot)),
    "standardized_rows": int(len(standardized_raw_panel)),
    "quote_filtered_rows": int(len(quote_filtered_panel)),
    "rejected_rows": int(len(quote_rejection_ledger)),
    "matched_pairs": int(len(matched_pairs_with_parity)),
    "candidate_rows_for_notebook09": int(len(notebook09_candidate_panel)),
    "candidate_expiries_for_notebook09": int(notebook09_candidate_panel["expiry"].nunique()),
    "parity_forward_outlier_rate": float(matched_pairs_with_parity["parity_forward_outlier"].mean()),
}

# ------------------------------------------------------------
# Save validation ledger and manifest
# ------------------------------------------------------------

final_validation_ledger_path = PROCESSED_DATA_DIR / f"{CONFIG.ticker.lower()}_notebook08_final_validation_ledger_{RUN_ID}.csv"
final_validation_ledger.to_csv(final_validation_ledger_path, index=False)

ARTIFACT_PATHS["final_validation_ledger"] = final_validation_ledger_path
MANIFEST["artifact_paths"]["final_validation_ledger"] = str(final_validation_ledger_path)

manifest_path = Path(ARTIFACT_PATHS["snapshot_manifest"])
manifest_path.parent.mkdir(parents=True, exist_ok=True)

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(json_safe(MANIFEST), file, indent=2)

ARTIFACT_PATHS["snapshot_manifest"] = manifest_path
MANIFEST["artifact_paths"]["snapshot_manifest"] = str(manifest_path)

print("Final Notebook 08 validation completed.")
print(f"NOTEBOOK_08_STATUS: {NOTEBOOK_08_STATUS}")
print(f"NOTEBOOK_08_READY_FOR_09: {NOTEBOOK_08_READY_FOR_09}")
print(f"Final validation pass count: {MANIFEST['final_validation_pass_count']}")
print(f"Final validation fail count: {MANIFEST['final_validation_fail_count']}")
print(f"Candidate rows for Notebook 09: {MANIFEST['final_handoff_summary']['candidate_rows_for_notebook09']:,}")
print(f"Candidate expiries for Notebook 09: {MANIFEST['final_handoff_summary']['candidate_expiries_for_notebook09']:,}")
print(f"Final validation ledger path: {final_validation_ledger_path}")
print(f"Manifest path: {manifest_path}")

display(final_validation_ledger)

if not NOTEBOOK_08_READY_FOR_09:
    print("Critical validation failures:")
    display(critical_failures)
else:
    print("Notebook 08 produced a quote-quality-controlled SPY option panel ready for Notebook 09.")
    print("This does not claim that the market surface is arbitrage-free.")

Final Notebook 08 validation completed.
NOTEBOOK_08_STATUS: READY_FOR_09
NOTEBOOK_08_READY_FOR_09: True
Final validation pass count: 29
Final validation fail count: 0
Candidate rows for Notebook 09: 3,527
Candidate expiries for Notebook 09: 12
Final validation ledger path: d:\Derivative Pricing Project v1.0+\V1.1\data\processed\spy_notebook08_final_validation_ledger_20260705_154512_UTC.csv
Manifest path: d:\Derivative Pricing Project v1.0+\V1.1\data\manifest\spy_option_snapshot_manifest_20260705_154512_UTC.json


,check_name,status,severity,observed_value,expected_value,description
0,artifact_exists__expiry_summary,PASS,CRITICAL,d:\Derivative Pricing Project v1.0+\V1.1\data\...,path exists,Required artifact must exist: expiry_summary.
1,artifact_exists__matched_pairs,PASS,CRITICAL,d:\Derivative Pricing Project v1.0+\V1.1\data\...,path exists,Required artifact must exist: matched_pairs.
2,artifact_exists__n09_candidate_panel,PASS,CRITICAL,d:\Derivative Pricing Project v1.0+\V1.1\data\...,path exists,Required artifact must exist: n09_candidate_pa...
3,artifact_exists__quote_filtered_panel,PASS,CRITICAL,d:\Derivative Pricing Project v1.0+\V1.1\data\...,path exists,Required artifact must exist: quote_filtered_p...
4,artifact_exists__raw_snapshot,PASS,CRITICAL,d:\Derivative Pricing Project v1.0+\V1.1\data\...,path exists,Required artifact must exist: raw_snapshot.
5,artifact_exists__raw_standardized_panel,PASS,CRITICAL,d:\Derivative Pricing Project v1.0+\V1.1\data\...,path exists,Required artifact must exist: raw_standardized...
6,artifact_exists__rejection_ledger,PASS,CRITICAL,d:\Derivative Pricing Project v1.0+\V1.1\data\...,path exists,Required artifact must exist: rejection_ledger.
7,at_least_one_expiry_passes_for_09,PASS,CRITICAL,12,>= 1,At least one expiry must pass the Notebook 09 ...
8,at_least_one_usable_forward_expiry,PASS,CRITICAL,12,>= 1,At least one expiry must have a usable parity-...
9,candidate_expiries_match_passed_expiries,PASS,CRITICAL,"[2026-07-06, 2026-07-07, 2026-07-08, 2026-07-0...","[2026-07-06, 2026-07-07, 2026-07-08, 2026-07-0...",Candidate panel should contain exactly the exp...


Notebook 08 produced a quote-quality-controlled SPY option panel ready for Notebook 09.
This does not claim that the market surface is arbitrage-free.
